In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install segmentation-models-pytorch albumentations==1.4.7 torchmetrics opencv-python

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# prompt: '/content/drive/MyDrive/Colab Notebooks/Root_seg/Dataset/images_266.tar'를 '/content' 아래에 압축을 풀어 줘

# Create the target directory if it doesn't exist

!tar -xf '/content/drive/MyDrive/Colab Notebooks/Root_seg/Dataset/Dataset_split_train_test_266.tar' -C '/content'

In [ ]:
# prompt: '/content/dataset/annotations.json'에 따라 mask 파일을 만들어 줘. 이 때 'Root'가 포함되어야 해
# prompt: 'Tooth'와 'Root'를 prediction하도록 수정해 줘

# Define paths
DATA_ROOT = '/content/content/Dataset_split_train_test_266/train'
ANNOTATIONS_PATH = os.path.join(DATA_ROOT, 'train_annotations.json') # Use the merged annotations
MASKS_DIR = os.path.join(DATA_ROOT, 'masks')
os.makedirs(MASKS_DIR, exist_ok=True) # Ensure masks directory exists


from PIL import Image, ImageDraw
import numpy as np
import os
import json

def coco_to_masks(json_path, output_mask_dir):
    """
    Converts COCO instance segmentation annotations (polygons) to semantic segmentation masks.

    Args:
        json_path (str): Path to the COCO annotations JSON file.
        output_mask_dir (str): Directory to save the generated mask images.
    """
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # Create a mapping from image ID to image info (width, height)
    image_info = {img['id']: (img['width'], img['height'], img['file_name']) for img in coco_data['images']}

    # Create a mapping from category name to category ID
    category_info = {cat['name']: cat['id'] for cat in coco_data['categories']}

    # Ensure output directory exists
    os.makedirs(output_mask_dir, exist_ok=True)

    # Process annotations
    for ann in coco_data['annotations']:
        image_id = ann['image_id']
        category_id = ann['category_id']
        segmentation = ann['segmentation']

        # Skip if image_id is not found (shouldn't happen with proper merging but good check)
        if image_id not in image_info:
            print(f"Warning: Image ID {image_id} not found in image info. Skipping annotation.")
            continue

        width, height, file_name = image_info[image_id]
        mask_file_name = os.path.splitext(file_name)[0] + '.png'
        mask_path = os.path.join(output_mask_dir, mask_file_name)

        # Get the existing mask if it exists, otherwise create a blank one
        # Ensure mask is in 'L' mode (grayscale)
        if os.path.exists(mask_path):
            mask_img = Image.open(mask_path).convert('L')
            mask_np = np.array(mask_img)
        else:
            mask_np = np.zeros((height, width), dtype=np.uint8)

        # Get category name from category_id
        # Find category name from category_info based on category_id
        category_name = None
        for name, cat_id in category_info.items():
            if cat_id == category_id:
                category_name = name
                break

        # Assign value based on category name - Tooth=1, Root=2
        mask_value = 0  # Default to background
        if category_name == 'Tooth':
            mask_value = 1 # Assign 1 for Tooth
        elif category_name == 'Root':
            mask_value = 2 # Assign 2 for Root


        # Only draw if it's not background and the category was found and is one of our targets
        if mask_value > 0 and category_name in ['Tooth', 'Root']:
            # Create a temporary mask for the current annotation
            annotation_mask_img = Image.new('L', (width, height), 0)
            draw = ImageDraw.Draw(annotation_mask_img)

            # Draw each polygon for the annotation
            for polygon in segmentation:
                # Ensure polygon coordinates are integers
                # Filter out non-numeric or malformed coordinates if necessary
                try:
                    # Flatten list of lists if necessary (COCO can have multi-polygon)
                    if isinstance(polygon[0], list):
                        flat_polygon = [item for sublist in polygon for item in sublist]
                    else:
                        flat_polygon = polygon

                    polygon_int = [int(p) for p in flat_polygon]
                    if len(polygon_int) % 2 != 0 or len(polygon_int) < 6: # Ensure even number of coordinates and at least 3 points (6 coords)
                         print(f"Skipping malformed or incomplete polygon in {file_name} for category {category_name}")
                         continue
                    # Draw the polygon on the annotation mask
                    # Note: draw.polygon expects a list of (x, y) tuples or a flattened list of coordinates
                    # It seems draw.polygon handles flattened list directly
                    draw.polygon(polygon_int, outline=mask_value, fill=mask_value)
                except (ValueError, TypeError) as e:
                    print(f"Skipping polygon due to error in {file_name} for category {category_name}: {e}")
                    continue


            # Combine the new mask with the existing mask using max operation
            # Max operation ensures that if Tooth and Root overlap, the higher value (Root=2) takes precedence
            # You might want a different logic for overlapping areas depending on your needs
            current_annotation_mask_np = np.array(annotation_mask_img)
            mask_np = np.maximum(mask_np, current_annotation_mask_np)

        # Save the updated mask after processing all annotations for the image
        # This is important: process all annotations for ONE image before saving its mask
        # The current loop processes annotations one by one across ALL images.
        # Need to process annotations image by image.

    # --- Revised Logic: Process annotations per image ---
    print("Processing annotations image by image...")
    processed_image_ids = set()

    for img in coco_data.get('images', []):
        image_id = img['id']
        width = img['width']
        height = img['height']
        file_name = img['file_name']
        mask_file_name = os.path.splitext(file_name)[0] + '.png'
        mask_path = os.path.join(output_mask_dir, mask_file_name)

        # Create a blank mask for the current image
        mask_np = np.zeros((height, width), dtype=np.uint8)

        # Find all annotations for this image
        image_annotations = [ann for ann in coco_data.get('annotations', []) if ann['image_id'] == image_id]

        if not image_annotations:
            # If no annotations for this image, save a blank mask (optional, or skip)
            # print(f"No annotations for image {file_name}. Saving blank mask.")
            mask_img_updated = Image.fromarray(mask_np)
            mask_img_updated.save(mask_path)
            processed_image_ids.add(image_id)
            continue # Move to the next image

        for ann in image_annotations:
            category_id = ann['category_id']
            segmentation = ann['segmentation']

            # Get category name
            category_name = None
            for name, cat_id in category_info.items():
                if cat_id == category_id:
                    category_name = name
                    break

            # Assign mask value
            mask_value = 0
            if category_name == 'Tooth':
                mask_value = 1
            elif category_name == 'Root':
                mask_value = 2

            if mask_value > 0 and category_name in ['Tooth', 'Root']:
                 annotation_mask_img = Image.new('L', (width, height), 0)
                 draw = ImageDraw.Draw(annotation_mask_img)

                 for polygon in segmentation:
                    try:
                        # Flatten list of lists if necessary
                        if isinstance(polygon[0], list):
                            flat_polygon = [item for sublist in polygon for item in sublist]
                        else:
                            flat_polygon = polygon

                        polygon_int = [int(p) for p in flat_polygon]
                        if len(polygon_int) % 2 != 0 or len(polygon_int) < 6:
                            print(f"Skipping malformed or incomplete polygon in {file_name} for category {category_name}")
                            continue

                        draw.polygon(polygon_int, outline=mask_value, fill=mask_value)
                    except (ValueError, TypeError) as e:
                        print(f"Skipping polygon due to error in {file_name} for category {category_name}: {e}")
                        continue

                 current_annotation_mask_np = np.array(annotation_mask_img)
                 # Use maximum to handle overlaps, preserving higher value (Root over Tooth if overlap)
                 mask_np = np.maximum(mask_np, current_annotation_mask_np)


        # Save the final mask for the current image
        mask_img_updated = Image.fromarray(mask_np)
        mask_img_updated.save(mask_path)
        processed_image_ids.add(image_id)
        print(f"Processed mask for {file_name}")


# Run the conversion
# Note: Using the merged annotations file
coco_to_masks(ANNOTATIONS_PATH, MASKS_DIR)

In [ ]:
# prompt: '/content/dataset/annotations.json'에 따라 mask 파일을 만들어 줘. 이 때 'Root'가 포함되어야 해
# prompt: 'Tooth'와 'Root'를 prediction하도록 수정해 줘

# Define paths
DATA_ROOT = '/content/content/Dataset_split_train_test_266/test'
ANNOTATIONS_PATH = os.path.join(DATA_ROOT, 'test_annotations.json') # Use the merged annotations
MASKS_DIR = os.path.join(DATA_ROOT, 'masks')
os.makedirs(MASKS_DIR, exist_ok=True) # Ensure masks directory exists


from PIL import Image, ImageDraw
import numpy as np
import os
import json

def coco_to_masks(json_path, output_mask_dir):
    """
    Converts COCO instance segmentation annotations (polygons) to semantic segmentation masks.

    Args:
        json_path (str): Path to the COCO annotations JSON file.
        output_mask_dir (str): Directory to save the generated mask images.
    """
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # Create a mapping from image ID to image info (width, height)
    image_info = {img['id']: (img['width'], img['height'], img['file_name']) for img in coco_data['images']}

    # Create a mapping from category name to category ID
    category_info = {cat['name']: cat['id'] for cat in coco_data['categories']}

    # Ensure output directory exists
    os.makedirs(output_mask_dir, exist_ok=True)

    # Process annotations
    for ann in coco_data['annotations']:
        image_id = ann['image_id']
        category_id = ann['category_id']
        segmentation = ann['segmentation']

        # Skip if image_id is not found (shouldn't happen with proper merging but good check)
        if image_id not in image_info:
            print(f"Warning: Image ID {image_id} not found in image info. Skipping annotation.")
            continue

        width, height, file_name = image_info[image_id]
        mask_file_name = os.path.splitext(file_name)[0] + '.png'
        mask_path = os.path.join(output_mask_dir, mask_file_name)

        # Get the existing mask if it exists, otherwise create a blank one
        # Ensure mask is in 'L' mode (grayscale)
        if os.path.exists(mask_path):
            mask_img = Image.open(mask_path).convert('L')
            mask_np = np.array(mask_img)
        else:
            mask_np = np.zeros((height, width), dtype=np.uint8)

        # Get category name from category_id
        # Find category name from category_info based on category_id
        category_name = None
        for name, cat_id in category_info.items():
            if cat_id == category_id:
                category_name = name
                break

        # Assign value based on category name - Tooth=1, Root=2
        mask_value = 0  # Default to background
        if category_name == 'Tooth':
            mask_value = 1 # Assign 1 for Tooth
        elif category_name == 'Root':
            mask_value = 2 # Assign 2 for Root


        # Only draw if it's not background and the category was found and is one of our targets
        if mask_value > 0 and category_name in ['Tooth', 'Root']:
            # Create a temporary mask for the current annotation
            annotation_mask_img = Image.new('L', (width, height), 0)
            draw = ImageDraw.Draw(annotation_mask_img)

            # Draw each polygon for the annotation
            for polygon in segmentation:
                # Ensure polygon coordinates are integers
                # Filter out non-numeric or malformed coordinates if necessary
                try:
                    # Flatten list of lists if necessary (COCO can have multi-polygon)
                    if isinstance(polygon[0], list):
                        flat_polygon = [item for sublist in polygon for item in sublist]
                    else:
                        flat_polygon = polygon

                    polygon_int = [int(p) for p in flat_polygon]
                    if len(polygon_int) % 2 != 0 or len(polygon_int) < 6: # Ensure even number of coordinates and at least 3 points (6 coords)
                         print(f"Skipping malformed or incomplete polygon in {file_name} for category {category_name}")
                         continue
                    # Draw the polygon on the annotation mask
                    # Note: draw.polygon expects a list of (x, y) tuples or a flattened list of coordinates
                    # It seems draw.polygon handles flattened list directly
                    draw.polygon(polygon_int, outline=mask_value, fill=mask_value)
                except (ValueError, TypeError) as e:
                    print(f"Skipping polygon due to error in {file_name} for category {category_name}: {e}")
                    continue


            # Combine the new mask with the existing mask using max operation
            # Max operation ensures that if Tooth and Root overlap, the higher value (Root=2) takes precedence
            # You might want a different logic for overlapping areas depending on your needs
            current_annotation_mask_np = np.array(annotation_mask_img)
            mask_np = np.maximum(mask_np, current_annotation_mask_np)

        # Save the updated mask after processing all annotations for the image
        # This is important: process all annotations for ONE image before saving its mask
        # The current loop processes annotations one by one across ALL images.
        # Need to process annotations image by image.

    # --- Revised Logic: Process annotations per image ---
    print("Processing annotations image by image...")
    processed_image_ids = set()

    for img in coco_data.get('images', []):
        image_id = img['id']
        width = img['width']
        height = img['height']
        file_name = img['file_name']
        mask_file_name = os.path.splitext(file_name)[0] + '.png'
        mask_path = os.path.join(output_mask_dir, mask_file_name)

        # Create a blank mask for the current image
        mask_np = np.zeros((height, width), dtype=np.uint8)

        # Find all annotations for this image
        image_annotations = [ann for ann in coco_data.get('annotations', []) if ann['image_id'] == image_id]

        if not image_annotations:
            # If no annotations for this image, save a blank mask (optional, or skip)
            # print(f"No annotations for image {file_name}. Saving blank mask.")
            mask_img_updated = Image.fromarray(mask_np)
            mask_img_updated.save(mask_path)
            processed_image_ids.add(image_id)
            continue # Move to the next image

        for ann in image_annotations:
            category_id = ann['category_id']
            segmentation = ann['segmentation']

            # Get category name
            category_name = None
            for name, cat_id in category_info.items():
                if cat_id == category_id:
                    category_name = name
                    break

            # Assign mask value
            mask_value = 0
            if category_name == 'Tooth':
                mask_value = 1
            elif category_name == 'Root':
                mask_value = 2

            if mask_value > 0 and category_name in ['Tooth', 'Root']:
                 annotation_mask_img = Image.new('L', (width, height), 0)
                 draw = ImageDraw.Draw(annotation_mask_img)

                 for polygon in segmentation:
                    try:
                        # Flatten list of lists if necessary
                        if isinstance(polygon[0], list):
                            flat_polygon = [item for sublist in polygon for item in sublist]
                        else:
                            flat_polygon = polygon

                        polygon_int = [int(p) for p in flat_polygon]
                        if len(polygon_int) % 2 != 0 or len(polygon_int) < 6:
                            print(f"Skipping malformed or incomplete polygon in {file_name} for category {category_name}")
                            continue

                        draw.polygon(polygon_int, outline=mask_value, fill=mask_value)
                    except (ValueError, TypeError) as e:
                        print(f"Skipping polygon due to error in {file_name} for category {category_name}: {e}")
                        continue

                 current_annotation_mask_np = np.array(annotation_mask_img)
                 # Use maximum to handle overlaps, preserving higher value (Root over Tooth if overlap)
                 mask_np = np.maximum(mask_np, current_annotation_mask_np)


        # Save the final mask for the current image
        mask_img_updated = Image.fromarray(mask_np)
        mask_img_updated.save(mask_path)
        processed_image_ids.add(image_id)
        print(f"Processed mask for {file_name}")


# Run the conversion
# Note: Using the merged annotations file
coco_to_masks(ANNOTATIONS_PATH, MASKS_DIR)

In [ ]:
# prompt: '/content/Dataset_split_train_test_266/test/masks' 폴더 아래 파일 중 random으로 10개에 대해 시각화해줘.

import random
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Directory containing the masks
mask_dir = '/content/content/Dataset_split_train_test_266/test/masks'

# Get a list of mask filenames
mask_filenames = os.listdir(mask_dir)

# Filter out non-PNG files if necessary (assuming masks are .png)
mask_filenames = [f for f in mask_filenames if f.endswith('.png')]

# Check if there are enough files to sample
if len(mask_filenames) < 10:
    print(f"Found only {len(mask_filenames)} mask files. Visualizing all of them.")
    sample_mask_filenames = mask_filenames
else:
    # Randomly sample 10 mask filenames
    sample_mask_filenames = random.sample(mask_filenames, 10)

for mask_filename in sample_mask_filenames:
    # Construct the full path for the mask
    mask_path = os.path.join(mask_dir, mask_filename)

    # Read the mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Display the mask
    plt.figure(figsize=(6, 6))
    plt.imshow(mask, cmap='gray') # Display mask in grayscale
    plt.title(f'Mask: {mask_filename}')
    plt.axis('off')
    plt.show()

In [ ]:
# prompt: '/content/dataset'폴더 아래 이미지와 mask를 중첩해서 10개 파일에 대해 보여 줘

import matplotlib.pyplot as plt
import os
import cv2 # Import cv2 here
import random # Import random

# Directory containing the images and masks
data_dir = '/content/content/Dataset_split_train_test_266/test'
image_dir = os.path.join(data_dir, 'images')
mask_dir = os.path.join(data_dir, 'masks')

# Get a list of image filenames (assuming images and masks have the same base name)
image_filenames = os.listdir(image_dir)
# Take the first 10 files # Changed to random sample 10 files
if len(image_filenames) < 10:
    print(f"Found only {len(image_filenames)} image files. Visualizing all of them.")
    sample_filenames = image_filenames
else:
    sample_filenames = random.sample(image_filenames, 10)


for filename in sample_filenames:
    # Construct the full path for the image and mask
    image_path = os.path.join(image_dir, filename)
    # Assuming mask filenames have the same base name but different extension (e.g., .png)
    mask_filename = os.path.splitext(filename)[0] + '.png'
    mask_path = os.path.join(mask_dir, mask_filename)

    # Check if both image and mask files exist
    if os.path.exists(image_path) and os.path.exists(mask_path):
        # Read the image
        image = cv2.imread(image_path)
        # Convert BGR to RGB for displaying with matplotlib
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read the mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Create an overlay: colorize the mask and blend it with the image
        # You can adjust the transparency (alpha) and colors as needed
        overlay = image_rgb.copy()
        # Example: Colorize Tooth (value 1) in green and Root (value 2) in blue
        overlay[mask == 1] = [0, 255, 0]  # Green for Tooth (changed from red)
        overlay[mask == 2] = [0, 0, 255]  # Blue for Root (kept blue)

        # Blend the original image and the overlay
        alpha = 0.2 # Transparency factor (0.0 for pure image, 1.0 for pure overlay)
        blended_image = cv2.addWeighted(image_rgb, 1 - alpha, overlay, alpha, 0)

        # Display the original image, mask, and blended image
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(image_rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray') # Display mask in grayscale
        axes[1].set_title('Mask')
        axes[1].axis('off')

        axes[2].imshow(blended_image)
        axes[2].set_title('Image with Mask Overlay')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print(f"Skipping {filename}: Image or mask not found.")

In [ ]:
# prompt: '/content/Dataset/masks'아래 있는 png 파일 갯수를 출력해 줘

!ls -1 '/content/content/Dataset_split_train_test_266/test/masks'/*.png | wc -l

In [ ]:
# unetpp_oral_kfold.py

# ---------------------------------------------------------------
# 5‑Fold CV for Tooth–Root segmentation with Unet++ (ResNet34, ImageNet)
# ---------------------------------------------------------------


import os, copy, time, json
from pathlib import Path
from glob import glob

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassJaccardIndex,
    MulticlassPrecision, MulticlassRecall, MulticlassF1Score
)

In [ ]:
# ---------------------- 0. 설정값 ----------------------
DATA_DIR    = Path('/content/Dataset_split_train_test_266/train')
IMG_SIZE    = 768 # Reduced image size to mitigate OutOfMemoryError # Changed IMG_SIZE to 768
BATCH_SIZE  = 6 # Reduced batch size to mitigate OutOfMemoryError # Changed batch size to 6
MAX_EPOCHS  = 150
PATIENCE  = 20
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3               # background + Tooth + Root
KFOLD       = 5
SEED        = 42

In [ ]:

# ---------------------- 1. Dataset ---------------------
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
        if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            aug = self.tf(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        mask = mask.squeeze().long()
        return img, mask

In [ ]:
# ---------------------- 4. Augmentations -------------------------
def get_train_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.ShiftScaleRotate(shift_limit=0.0,
                           scale_limit=0.10,         # ±10 %
                           rotate_limit=15,          # ±15°
                           shear_factor=0.05,        # Use shear_factor instead of shear_limit
                           p=0.9),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.0, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

In [ ]:
# ---------------------- 5. Metrics -------------------------------
from torchmetrics.classification import MulticlassJaccardIndex # Import JaccardIndex here

def build_metrics():
    # Update num_classes to N_CLASSES (3)
    metric_kwargs = dict(num_classes=N_CLASSES, average=None, ignore_index=None)
    metrics = {
        'acc' : MulticlassAccuracy(**metric_kwargs),
        'iou' : MulticlassJaccardIndex(**metric_kwargs), # Use MulticlassJaccardIndex
        'prec': MulticlassPrecision(**metric_kwargs),
        'rec' : MulticlassRecall(**metric_kwargs),
        'f1'  : MulticlassF1Score(**metric_kwargs),
    }
    # Move metrics to the specified device
    return {k: v.to(DEVICE) for k, v in metrics.items()}

def update_metrics(metric_dict, logits, y): # Changed preds to logits to match evaluate functions
    # preds = preds.argmax(dim=1) # argmax is done inside the metric.update for most metrics
    # Fix: Rely on build_metrics to move metrics to device, remove redundant .to(DEVICE) in loop
    for m in metric_dict.values():
        # Ensure metric is on the correct device before updating - Should be handled by build_metrics
        # Removed: m = m.to(DEVICE)
        m.update(logits, y) # Pass logits and target to metric update

def compute_metric_dict(metric_dict):
    return {k: v.compute().cpu().numpy() for k, v in metric_dict.items()}

def reset_metrics(metric_dict):
    for m in metric_dict.values():
        m.reset()

In [ ]:
# ---------------------- 5. Train one Fold -------------
def train_one_fold(fold, train_idx, val_idx, img_paths, mask_paths):
    print(f"\n==========  Fold {fold+1}/{KFOLD}  ==========")
    # Dataset / Loader
    tr_ds = OralSegDataset([img_paths[i]  for i in train_idx],
                           [mask_paths[i] for i in train_idx],
                           transforms=get_train_aug())
    va_ds = OralSegDataset([img_paths[i]  for i in val_idx],
                           [mask_paths[i] for i in val_idx],
                           transforms=get_val_aug())
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=4, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=4, pin_memory=True)

    # Model / Optim / Sched
    model = build_model().to(DEVICE)
    loss_fn = criterion()

    dec_params  = [p for n, p in model.named_parameters()
                     if 'decoder' in n or 'segmentation_head' in n]
    enc_params  = [p for n, p in model.named_parameters() if 'encoder' in n]
    optimizer = torch.optim.AdamW([
        {'params': enc_params, 'lr': 5e-5},
        {'params': dec_params, 'lr': 1e-4}
    ], weight_decay=1e-4)

    # Initial scheduler: CosineAnnealingWarmRestarts
    initial_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6)

    # Define the epoch to switch schedulers (e.g., after 2 cycles of T_0=20)
    switch_epoch = 2 * 20 # Switch after the first two cycles

    # Secondary scheduler (e.g., standard CosineAnnealingLR)
    # You could also use ReduceLROnPlateau here
    secondary_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS - switch_epoch, eta_min=1e-7) # T_max is remaining epochs

    scheduler = initial_scheduler # Start with the initial scheduler


    best_iou, patience_cnt, best_w = -1, 0, None
    metrics_tr, metrics_va = build_metrics(), build_metrics()

    # Lists to store per-epoch metrics for plotting
    epoch_va_iou = {cls_name: [] for cls_name in ['Background', 'Tooth', 'Root']}


    for epoch in range(1, MAX_EPOCHS+1):
        t0 = time.time()

        # Switch scheduler if criteria met
        if epoch == switch_epoch + 1: # Switch at the beginning of the epoch after switch_epoch
            print(f"Switching scheduler to CosineAnnealingLR at epoch {epoch}")
            scheduler = secondary_scheduler
            # Reset patience counter maybe? Depends on desired behavior
            # patience_cnt = 0


        # ----- Train -----
        model.train(); reset_metrics(metrics_tr); tr_loss = 0
        for x, y, _ in tr_dl:
            x,y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            # Pass original y (Long type) to loss function
            loss = loss_fn(out, y); loss.backward(); optimizer.step()
            tr_loss += loss.item()*x.size(0)
            # Use original y (Long type) for metrics
            update_metrics(metrics_tr, out, y)

        # ----- Val -----
        model.eval(); reset_metrics(metrics_va); va_loss = 0
        with torch.no_grad():
            for x, y, _ in va_dl:
                x,y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                # Pass original y (Long type) to loss function
                va_loss += loss_fn(out, y).item()*x.size(0)
                # Use original y (Long type) for metrics
                update_metrics(metrics_va, out, y)

        tr_loss /= len(tr_ds); va_loss /= len(va_ds)
        va_scores = compute_metric_dict(metrics_va)
        # Calculate mean IoU for Tooth and Root (indices 1 and 2)
        iou_val   = va_scores['iou'][1:N_CLASSES].mean()

        # Step the current scheduler
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
             # For ReduceLROnPlateau, step based on validation metric
             scheduler.step(iou_val)
        else:
             # For other schedulers like CosineAnnealing, step each epoch
             scheduler.step()


        # Store per-class IoU for plotting
        class_names = ['Background', 'Tooth', 'Root']
        for cls_id, cls_name in enumerate(class_names):
             if cls_id < len(va_scores['iou']):
                  epoch_va_iou[cls_name].append(va_scores['iou'][cls_id])


        print(f"[E{epoch:03d}] tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} va_mIoU={iou_val:.4f} Δt={time.time()-t0:.1f}s")

        # Early‑Stopping based on validation mIoU
        if iou_val > best_iou:
            best_iou, patience_cnt, best_w = iou_val, 0, copy.deepcopy(model.state_dict())
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print("> early stop"); break

    # ---- Save best model for this fold ----
    mpath = f'unetpp_fold{fold+1}_best.pth'
    torch.save(best_w, mpath)
    print(f"Fold {fold+1}: best mIoU={best_iou:.4f}  (saved → {mpath})")

    # ---- Final metrics on val (best model) ----
    model.load_state_dict(best_w); model.eval()
    reset_metrics(metrics_va)
    with torch.no_grad():
        for x, y, _ in va_dl:
            x,y = x.to(DEVICE), y.to(DEVICE)
            update_metrics(metrics_va, model(x), y)
    va_scores = compute_metric_dict(metrics_va)   # dict -> arrays

    # Organise to dict with names
    # Updated class names to reflect 3 classes
    names = ['Background', 'Tooth', 'Root']
    results = []
    for cls_id, cls_name in enumerate(names):
        for m, arr in va_scores.items():
            val = arr[cls_id] # Get metric for the specific class
            results.append({
                'fold': fold+1,
                'class': cls_name,
                'metric': m,
                'value': float(val)
            })

    # Add Overall mIoU (mean of Tooth and Root IoU)
    overall_iou = va_scores['iou'][1:N_CLASSES].mean()
    results.append({
        'fold': fold+1,
        'class': 'Overall',
        'metric': 'iou',
        'value': float(overall_iou)
    })
     # Add Overall f1 (mean of Tooth and Root f1)
    overall_f1 = va_scores['f1'][1:N_CLASSES].mean()
    results.append({
        'fold': fold+1,
        'class': 'Overall',
        'metric': 'f1',
        'value': float(overall_f1)
    })
    # Add Overall accuracy (pixel accuracy over all classes)
    overall_acc = va_scores['acc'][0] # Overall pixel accuracy is the first element of the accuracy array
    results.append({
        'fold': fold+1,
        'class': 'Overall',
        'metric': 'acc',
        'value': float(overall_acc)
    })
     # Add Overall precision (mean of Tooth and Root precision)
    overall_prec = va_scores['prec'][1:N_CLASSES].mean()
    results.append({
        'fold': fold+1,
        'class': 'Overall',
        'metric': 'prec',
        'value': float(overall_prec)
    })
     # Add Overall recall (mean of Tooth and Root recall)
    overall_rec = va_scores['rec'][1:N_CLASSES].mean()
    results.append({
        'fold': fold+1,
        'class': 'Overall',
        'metric': 'rec',
        'value': float(overall_rec)
    })


    return results

In [ ]:
# ---------------------- 4. Model / Loss ----------------
import torch.nn as nn
import segmentation_models_pytorch as smp
import torch

# Need to implement or import Lovasz-Softmax loss
# A common implementation can be found online or in segmentation libraries
# For this example, let's assume we have a function called 'lovasz_softmax' available

# For demonstration, let's use a placeholder or a simplified version if a full implementation is complex
# A common source for Lovasz loss is: https://github.com/bermanmaxim/LovaszSoftmax

# Let's define a simple placeholder or assume it's available
# If not available, this would require adding a new cell with the implementation.
# For now, let's assume it's imported or defined elsewhere.
# from your_lovasz_loss_file import lovasz_softmax # Example import

# --- Placeholder for lovasz_softmax function (replace with actual implementation) ---
# NOTE: A proper implementation of Lovasz-Softmax is required for this to work correctly.
# This is a simplified placeholder and might not be accurate.
# You would typically install a library or copy the implementation.
# Example (simplified):
# def lovasz_softmax(probas, labels, classes='present', per_image=False, ignore=None):
#     # This is a placeholder. Replace with a real implementation.
#     # See https://github.com/bermanmaxim/LovaszSoftmax for a correct implementation
#     print("Using placeholder lovasz_softmax - replace with real implementation!")
#     # Dummy return value for placeholder
#     if probas.dim() == labels.dim() + 1:
#          probas = probas.argmax(dim=1) # Convert probabilities to predictions if needed
#     return torch.mean((probas != labels).float()) # Example: simple pixel error as placeholder
# -----------------------------------------------------------------------------------

# Let's assume a correct `lovasz_softmax` function is available.
# For Colab, we might need to install a library or add the implementation.
# As we cannot add new cells in modify_cells, let's assume a library is installed
# that provides `smp.losses.LovaszLoss`. If not, we'd need a separate step to add it.

# Check if smp.losses.LovaszLoss exists (it does in recent versions of smp)
if hasattr(smp.losses, 'LovaszLoss'):
    LovaszLoss = smp.losses.LovaszLoss
    print("Using smp.losses.LovaszLoss")
else:
     # If smp.losses.LovaszLoss is not available, we would need to add its implementation.
     # Since we are in modify_cells, we cannot add a new cell.
     # I will proceed assuming it's available or will be added in a separate step.
     print("smp.losses.LovaszLoss not found. Proceeding, but you might need to add its implementation.")
     # Define a dummy class to avoid errors if it's truly missing, but training will fail.
     class LovaszLoss(torch.nn.Module):
         def __init__(self, mode, ignore_index=None):
              super().__init__()
              print("Using Dummy LovaszLoss - replace with real implementation!")
              self.mode = mode
              self.ignore_index = ignore_index
         def forward(self, probas, labels):
              # Dummy loss
              if probas.dim() == labels.dim() + 1:
                   probas = probas.argmax(dim=1)
              return torch.mean((probas != labels).float()) # Example: simple pixel error


def build_model():
    # Modified to predict background, Tooth, and Root
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES, # Use N_CLASSES (3)
        in_channels    = 3,
        activation     = None # No activation for multiclass output before loss
    )

def criterion():
    # Use DiceLoss and LovaszLoss for multiclass segmentation
    # Note: LovaszLoss in smp expects logits, not probabilities
    dice = smp.losses.DiceLoss(mode='multiclass', ignore_index=None)
    # Lovasz loss from smp expects logits for 'multiclass' mode
    lovasz = LovaszLoss(mode='multiclass', ignore_index=None)


    # Adjust weights: you can experiment with these weights
    # Combine Lovasz and Dice loss
    return lambda logits, y: 0.5 * dice(logits, y) + 0.5 * lovasz(logits, y)

In [ ]:
# ---------------------- 6. Main ------------------------
def main():
    img_paths  = sorted(glob(str(DATA_DIR/'images'/'*')))
    mask_paths = sorted(glob(str(DATA_DIR/'masks'/'*')))
    #assert len(img_paths)==len(mask_paths), "이미지/마스크 수 불일치"
    print(f"Total images: {len(img_paths)}")

    kf = KFold(n_splits=KFOLD, shuffle=True, random_state=SEED)
    all_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(img_paths)):
        res = train_one_fold(fold, train_idx, val_idx, img_paths, mask_paths)
        all_results.extend(res)

    # Save CSV
    df = pd.DataFrame(all_results)
    csv_path = 'kfold_metrics.csv'
    df.to_csv(csv_path, index=False)
    print(f"\n=== 5‑Fold 결과 저장 완료 → {csv_path} ===")
    print(df.groupby(['class','metric'])['value'].mean().unstack().round(4))

if __name__ == "__main__":
    main()

In [ ]:
# prompt: '/content/kfold_metrics.csv'파일을 dataframe으로 읽어서 각 index를 column으로 위치시켜줘

df_metrics = pd.read_csv('/content/kfold_metrics.csv')

# 피벗 테이블을 사용하여 index를 column으로 재구성
# 'class'와 'metric'을 새로운 column으로 만들고, 'value'를 값으로 채웁니다.
df_pivot = df_metrics.pivot_table(index='fold', columns=['class', 'metric'], values='value')

# 필요에 따라 컬럼 이름을 정돈합니다 (예: ('Overall', 'acc') -> 'Overall_acc')
df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns]

# 인덱스를 리셋하여 'fold'를 일반 컬럼으로 만듭니다.
df_pivot = df_pivot.reset_index()

df_pivot

In [ ]:
# prompt: '/content/kfold_metrics.csv'파일을 dataframe으로 읽어서 각 index를 column으로 위치시켜줘

df_metrics = pd.read_csv('/content/kfold_metrics.csv')

# 피벗 테이블을 사용하여 index를 column으로 재구성
# 'class'와 'metric'을 새로운 column으로 만들고, 'value'를 값으로 채웁니다.
df_pivot = df_metrics.pivot_table(index='fold', columns=['class', 'metric'], values='value')

# 필요에 따라 컬럼 이름을 정돈합니다 (예: ('Overall', 'acc') -> 'Overall_acc')
df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns]

# 인덱스를 리셋하여 'fold'를 일반 컬럼으로 만듭니다.
df_pivot = df_pivot.reset_index()

# df_pivot를 csv파일로 저장해 줘
df_pivot.to_csv('/content/kfold_metrics.csv', index=False)
print("df_pivot saved to kfold_metrics.csv")

In [ ]:
# prompt: '/content/kfold_metrics.csv' 를 이용해 95%ci를 게산해 줘

import pandas as pd
import numpy as np
from scipy import stats

# Function to calculate confidence interval and return as a tuple
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    if n <= 1:
        return np.nan, np.nan, np.nan # mean, lower, upper
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

# Load the kfold metrics CSV
try:
    df_metrics_pivot = pd.read_csv('/content/kfold_metrics.csv')

    # Drop the 'fold' column as we calculate across folds
    df_values = df_metrics_pivot.drop(columns=['fold'])

    results_list = []

    # Iterate through columns (each column is a class_metric)
    for col in df_values.columns:
        # Extract class and metric name from column name (e.g., 'Background_acc')
        parts = col.split('_')
        if len(parts) >= 2:
            # Assume format is Class_Metric (e.g., Tooth_iou) or Overall_Metric
            # Handle cases like 'Background_acc', 'Overall_iou', 'Tooth_f1', 'Root_rec'
            # Check for 'Overall' first, then assume Class_Metric
            if parts[0] == 'Overall':
                 cls = 'Overall'
                 metric = '_'.join(parts[1:]) # Metric could be 'acc', 'iou', 'f1', etc.
            else:
                 cls = parts[0]
                 metric = '_'.join(parts[1:]) # Metric could be 'acc', 'iou', 'f1', etc.

            values = df_values[col].dropna() # Get values and drop NaNs for CI calculation

            # Calculate mean and CI
            mean_val, ci_lower, ci_upper = mean_confidence_interval(values)

            results_list.append({
                'class': cls,
                'metric': metric,
                'mean': mean_val,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            })
        else:
             print(f"Skipping column with unexpected format: {col}")


    mean_ci_results = pd.DataFrame(results_list)

    print("5-Fold Cross-Validation Results (Mean ± 95% CI):")
    # Print results in a readable format, sorted by class and metric
    mean_ci_results_sorted = mean_ci_results.sort_values(by=['class', 'metric'])

    for index, row in mean_ci_results_sorted.iterrows():
        cls = row['class']
        metric = row['metric']
        mean_val = row['mean']
        ci_lower = row['ci_lower']
        ci_upper = row['ci_upper']

        if np.isnan(ci_lower) or np.isnan(ci_upper):
             ci_str = " (CI: N/A)"
        else:
             ci_str = f" (CI: {ci_lower:.4f} - {ci_upper:.4f})"

        print(f"  Class: {cls:10s}, Metric: {metric:5s}, Mean: {mean_val:.4f}{ci_str}")

    # Optional: Save these results to a new CSV
    output_csv_path = '/content/kfold_mean_ci_metrics.csv'
    mean_ci_results.to_csv(output_csv_path, index=False)
    print(f"\nMean and CI results saved to {output_csv_path}")

except FileNotFoundError:
    print("Error: kfold_metrics.csv not found. Please run the training first to generate this file.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc() # Print detailed traceback for debugging

# Model Test

In [ ]:
# '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth'를 이용해서 '/content/Dataset_splited/test'에 대한 성능지표의 95%ci를 계산해 줘

from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score, MulticlassSpecificity
)
import torch
import numpy as np
import pandas as pd
import os
from glob import glob
import cv2
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy import stats # for confidence interval
from pathlib import Path # Import Path here


# Define constants (ensure these match your training constants)
IMG_SIZE    = 736
BATCH_SIZE  = 1 # Use batch size 1 for individual image metrics
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3               # background + Root (0, 1) # Corrected: background + Tooth + Root (0, 1, 2)

# Re-define the OralSegDataset and data augmentation (make sure they are consistent)
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
        if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            aug = self.tf(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        mask = mask.squeeze().long()
        # Return the original image path along with tensors
        return img, mask, self.imgs[idx]


def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# Re-define the model building function
def build_model():
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES,
        in_channels    = 3,
        activation     = None
    )

# Function to calculate Dice score manually
def calculate_dice(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

# Function to calculate IoU manually
def calculate_iou(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

# Function to calculate F1 score manually
def calculate_f1(pred, target, smooth=1e-6):
    tp = np.sum((pred == 1) & (target == 1))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))

    # Handle division by zero for precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Handle division by zero for F1 score
    f1 = (2. * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1

def calculate_ap(pred, target, iou_threshold=0.5):
    """
    Calculate Average Precision (AP) for a single class.
    """
    iou = calculate_iou(pred, target)
    if iou >= iou_threshold:
        return 1.0
    else:
        return 0.0


# Function to calculate confidence interval
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    if n <= 1:
        # Cannot compute CI with 1 or fewer samples
        return np.nan, np.nan, np.nan
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

# --- Evaluation on Test Set ---
def evaluate_test_set_and_collect_metrics(model_path, test_image_dir, test_mask_dir):
    """
    Loads a trained model, performs inference on the test set,
    and collects per-image metrics (Dice, IoU, Precision, Recall, Specificity, Accuracy, F1)
    into numpy arrays. Also computes 95% CI. Returns image filenames and Root mask areas.

    Args:
        model_path (str): Path to the trained model file (.pth).
        test_image_dir (str): Directory containing the test images.
        test_mask_dir (str): Directory containing the ground truth test masks.

    Returns:
        tuple: A tuple containing:
            - dict: A dictionary containing numpy arrays for each metric (Dice, IoU, etc.)
                    for each class ('Overall', 'Background', 'Tooth', 'Root').
            - dict: A dictionary for mean metrics.
            - dict: A dictionary for 95% CI for each metric/class.
            - list: A list of processed image filenames.
            - list: A list of corresponding 'Tooth' mask areas.
            - list: A list of corresponding 'Root' mask areas.
    """
    print("\n--- Evaluating on Test Set and Collecting Metrics ---")

    # Build model and load weights
    model = build_model().to(DEVICE)
    # Fix: Add weights_only=True to torch.load for safety
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    # Get test image and mask paths
    test_img_paths  = sorted(glob(str(Path(test_image_dir) / '*')))
    test_mask_paths = sorted(glob(str(Path(test_mask_dir) / '*')))

    if not test_img_paths or not test_mask_paths:
        print("Error: No test images or masks found in the specified directories.")
        return {}, {}, {}, [], [], [] # Return empty dictionaries and lists, adjusted for two mask areas

    assert len(test_img_paths) == len(test_mask_paths), "Test images and masks count mismatch."
    print(f"Found {len(test_img_paths)} test images and masks.")

    # Create a dataset for the test set
    test_transforms = get_val_aug()
    # Ensure OralSegDataset returns image path, tooth_mask_area, root_mask_area
    # Note: Calculating mask area here might be slow if done on the fly.
    # It might be better to calculate areas separately or load pre-calculated areas.
    # For simplicity in this fix, let's assume mask area calculation is handled efficiently by the dataset or we accept the potential overhead.
    test_dataset = OralSegDataset(test_img_paths, test_mask_paths, transforms=test_transforms)

    # Create a DataLoader for the test set
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)

    # Initialize lists to store per-image metrics and additional info
    all_metrics = {
        'dice': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'iou': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'prec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'rec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'spec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'acc': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'f1': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added F1 score
        'mAP50': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added mAP50 score
    }
    processed_filenames = []
    tooth_mask_areas = []
    root_mask_areas = []


    # --- Inference and Evaluation ---
    with torch.no_grad():
        # Unpack image, mask, and image path from the DataLoader
        for i, (x, y, img_path) in enumerate(test_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Store filename (get filename from path)
            processed_filenames.append(os.path.basename(img_path[0])) # Batch size is 1

            # Calculate Tooth and Root mask areas (on CPU)
            tooth_area = (y.cpu() == 1).sum().item()
            root_area = (y.cpu() == 2).sum().item() # Root is now class 2
            tooth_mask_areas.append(tooth_area)
            root_mask_areas.append(root_area)


            # Perform inference
            logits = model(x)
            preds = logits.argmax(dim=1) # Get predicted class index

            # Move predicted and target tensors to CPU for NumPy calculations
            preds_np = preds.cpu().numpy()
            y_np = y.cpu().numpy()

            dice_scores = []
            iou_scores = []
            acc_scores_np = []
            prec_scores_np = []
            rec_scores_np = []
            spec_scores_np = []
            f1_scores_np = [] # Added F1 score list
            ap50_scores_np = [] # Added AP50 score list

            # Calculate Dice, IoU, Accuracy, Precision, Recall, Specificity, F1 for each class using numpy
            for cls_id in range(N_CLASSES):
                pred_cls = (preds_np == cls_id).astype(np.uint8)
                target_cls = (y_np == cls_id).astype(np.uint8)

                # Calculate Dice and IoU (manual calculation is fine)
                # Avoid division by zero if a class is not present in the ground truth or prediction for this image
                if target_cls.sum() > 0 or pred_cls.sum() > 0:
                     dice = calculate_dice(pred_cls, target_cls)
                     iou = calculate_iou(pred_cls, target_cls)
                     dice_scores.append(dice)
                     iou_scores.append(iou)
                else:
                     # If neither predicted nor ground truth pixels for this class, metric is typically 1.0 (perfect score as there's nothing to segment)
                     # Append 1.0 as per convention for empty ground truth/prediction
                     dice_scores.append(1.0)
                     iou_scores.append(1.0)


                # Calculate Accuracy, Precision, Recall, Specificity, F1 using numpy
                tp = np.sum((pred_cls == 1) & (target_cls == 1))
                fp = np.sum((pred_cls == 1) & (target_cls == 0))
                tn = np.sum((pred_cls == 0) & (target_cls == 0))
                fn = np.sum((pred_cls == 0) & (target_cls == 1))

                # Calculate metrics (handle division by zero) - append NaN if denominator is zero
                accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan
                precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
                recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan # Also known as Sensitivity
                specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
                f1 = calculate_f1(pred_cls, target_cls) # Calculate F1 using the new function
                ap50 = calculate_ap(pred_cls, target_cls)

                acc_scores_np.append(accuracy)
                prec_scores_np.append(precision)
                rec_scores_np.append(recall)
                spec_scores_np.append(specificity)
                f1_scores_np.append(f1) # Added F1 score
                ap50_scores_np.append(ap50) # Added AP50 score


            # Store per-class metrics in the dictionary
            class_indices = {0: 'Background', 1: 'Tooth', 2: 'Root'} # Updated class indices
            metric_values = {
                'dice': dice_scores,
                'iou': iou_scores,
                'prec': prec_scores_np,
                'rec': rec_scores_np,
                'spec': spec_scores_np,
                'acc': acc_scores_np,
                'f1': f1_scores_np, # Added F1 score
                'mAP50': ap50_scores_np,
            }

            for metric_name, values in metric_values.items():
                for cls_id, cls_name in class_indices.items():
                    if cls_id < len(values):
                         all_metrics[metric_name][cls_name].append(values[cls_id])
                    else:
                         # This case should ideally not happen if values list has N_CLASSES elements
                         all_metrics[metric_name][cls_name].append(np.nan)


            # Calculate "Overall" metrics (mean over Tooth and Root) for this image
            # Append to the Overall list - use nanmean to handle NaNs from classes not present
            # Note: If both Tooth and Root are not present and metrics are 1.0, the mean will be 1.0.
            # If one is present and the other not (1.0), mean is average. If both NaN, mean is NaN.
            all_metrics['dice']['Overall'].append(np.nanmean([dice_scores[1], dice_scores[2]])) # mean of Tooth and Root Dice
            all_metrics['iou']['Overall'].append(np.nanmean([iou_scores[1], iou_scores[2]])) # mean of Tooth and Root IoU
            all_metrics['prec']['Overall'].append(np.nanmean([prec_scores_np[1], prec_scores_np[2]])) # mean of Tooth and Root Precision
            all_metrics['rec']['Overall'].append(np.nanmean([rec_scores_np[1], rec_scores_np[2]])) # mean of Tooth and Root Recall
            all_metrics['spec']['Overall'].append(np.nanmean([spec_scores_np[1], spec_scores_np[2]])) # mean of Tooth and Root Specificity
            # Overall pixel accuracy for the whole image - this is typically the Background accuracy when average=None
            all_metrics['acc']['Overall'].append(acc_scores_np[0]) # Background pixel accuracy as overall accuracy
            all_metrics['f1']['Overall'].append(np.nanmean([f1_scores_np[1], f1_scores_np[2]])) # mean of Tooth and Root F1
            all_metrics['mAP50']['Overall'].append(np.nanmean([ap50_scores_np[1], ap50_scores_np[2]]))


            print(f"Processed test image {i+1}/{len(test_img_paths)}")


    # Convert lists to numpy arrays
    metrics_np_arrays = {}
    for metric_name, classes_data in all_metrics.items():
        metrics_np_arrays[metric_name] = {}
        for cls_name, values_list in classes_data.items():
            metrics_np_arrays[metric_name][cls_name] = np.array(values_list)

    # Calculate Mean and 95% Confidence Interval
    mean_metrics = {}
    ci_metrics = {} # Stores (mean, lower_bound, upper_bound)
    for metric_name, classes_data in metrics_np_arrays.items():
        mean_metrics[metric_name] = {}
        ci_metrics[metric_name] = {}
        for cls_name, values_array in classes_data.items():
            # Calculate mean ignoring NaNs
            mean_metrics[metric_name][cls_name] = np.nanmean(values_array)
            # Calculate CI ignoring NaNs - pass only non-NaN values to mean_confidence_interval
            ci_metrics[metric_name][cls_name] = mean_confidence_interval(values_array[~np.isnan(values_array)])


    print("\n--- Test Set Evaluation Summary ---")
    print("Metrics (Mean ± 95% CI):")
    class_order = ['Overall', 'Background', 'Tooth', 'Root'] # Updated order for printing

    for cls_name in class_order:
         print(f"\nClass: {cls_name}")
         for metric_name in all_metrics.keys():
             mean_val = mean_metrics[metric_name][cls_name]
             ci_info = ci_metrics[metric_name][cls_name]

             # Format CI string (handle cases with not enough data for CI)
             if len(ci_info) == 3 and not np.isnan(ci_info[1]) and not np.isnan(ci_info[2]):
                 ci_str = f" (CI: {ci_info[1]:.4f} - {ci_info[2]:.4f})"
             else:
                 ci_str = " (CI: N/A)"

             print(f"  {metric_name.upper():<5s}: {mean_val:.4f}{ci_str}")

    # --- Save Mean and CI to CSV ---
    output_data = []
    for metric_name in all_metrics.keys():
        for cls_name in class_order:
            mean_val = mean_metrics[metric_name][cls_name]
            ci_info = ci_metrics[metric_name][cls_name]
            output_data.append({
                'Class': cls_name,
                'Metric': metric_name,
                'Mean': mean_val,
                'CI_Lower': ci_info[1] if len(ci_info) == 3 else np.nan,
                'CI_Upper': ci_info[2] if len(ci_info) == 3 else np.nan
            })

    df_mean_ci = pd.DataFrame(output_data)
    output_csv_path = '/content/test_set_mean_ci_metrics.csv'
    df_mean_ci.to_csv(output_csv_path, index=False)
    print(f"\nMean and CI results saved to {output_csv_path}")


    # Return Tooth and Root mask areas and filenames
    return metrics_np_arrays, mean_metrics, ci_metrics, processed_filenames, tooth_mask_areas, root_mask_areas


# Define paths for the test set evaluation
# Use fold 5 model as it is likely the best based on the training logs
model_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth' # Use the fold 5 best model
test_image_directory = '/content/content/Dataset_split_train_test_266/test/images'
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks'

# Run the evaluation and get the results as numpy arrays and CI, filenames, and mask areas
metrics_np_arrays, mean_metrics, ci_metrics, test_filenames, test_tooth_mask_areas, test_root_mask_areas = evaluate_test_set_and_collect_metrics(
    model_file_path, test_image_directory, test_mask_directory
)

# Now you have the metrics stored in numpy arrays in `metrics_np_arrays`,
# the mean and CI in `mean_metrics` and `ci_metrics`,
# the processed filenames in `test_filenames`, and the root mask areas in `test_root_mask_areas`.
# For example, the Dice scores for the Root class for each image are in:
# metrics_np_arrays['dice']['Root']

# The corresponding filenames are in `test_filenames` and root mask areas in `test_root_mask_areas`
# You can use these lists to analyze outlier cases based on their filenames and mask areas.

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
try:
    df_test_metrics = pd.read_csv('/content/test_set_mean_ci_metrics.csv')

    # Remove rows where the 'Class' column is 'Background'
    df_filtered_metrics = df_test_metrics[df_test_metrics['Class'] != 'Background'].copy()

    # Sort the DataFrame by the 'Class' column
    df_filtered_metrics_sorted = df_filtered_metrics.sort_values(by='Class').reset_index(drop=True)


    # Display the filtered and sorted DataFrame
    print("Metrics excluding Background class (sorted by Class):")
    display(df_filtered_metrics_sorted)

except FileNotFoundError:
    print("Error: /content/test_set_mean_ci_metrics.csv not found. Please ensure the test set evaluation has been run.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Save the filtered and sorted DataFrame to a CSV file
output_csv_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare/Unet_metrics.csv'
df_filtered_metrics_sorted.to_csv(output_csv_path, index=False)

print(f"Filtered and sorted metrics saved to {output_csv_path}")

In [ ]:
# Define paths
model_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth' # Use the fold 5 best model
test_image_directory = '/content/content/Dataset_split_train_test_266/test/images'
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks'

In [ ]:
import matplotlib.pyplot as plt
import os
import cv2
import torch
import torch.nn.functional as F # Import torch.nn.functional
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pathlib import Path
import numpy as np

# Assuming constants and model building functions (IMG_SIZE, BATCH_SIZE, DEVICE, ENCODER, ENC_WEIGHTS, N_CLASSES,
# OralSegDataset, get_val_aug, build_model) are defined in previous cells and are available in the current runtime.
# If they are not, you would need to redefine them here or ensure the user runs the preceding cells.

# Define constants (ensure these match your training constants)
IMG_SIZE    = 768 # Changed IMG_SIZE for test inference
BATCH_SIZE  = 1 # Use batch size 1 for visualization
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3 # background + Tooth + Root

# Re-define the OralSegDataset and data augmentation (make sure they are consistent)
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths # masks can be None for prediction
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)

        mask = None
        if self.masks:
             mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
             if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            # Apply transformations to image and mask (if mask exists)
            if mask is not None:
                aug = self.tf(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
                mask = mask.squeeze().long() # Ensure mask is squeezed and long type
            else:
                 # Apply transformations only to image
                 aug = self.tf(image=img)
                 img = aug['image']


        return img, mask, self.imgs[idx] # Return image, mask (can be None), and image path


def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# Re-define the model building function
def build_model():
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES,
        in_channels    = 3,
        activation     = None # No activation for multiclass output before loss/argmax
    )


# Define directory to save predicted masks
PREDICTED_MASKS_DIR = '/content/content/Dataset_split_train_test_266/test/predicted_masks'
os.makedirs(PREDICTED_MASKS_DIR, exist_ok=True)
print(f"Predicted masks will be saved to: {PREDICTED_MASKS_DIR}")


# Check if required variables are defined
if 'model_file_path' not in locals() or 'test_image_directory' not in locals() or 'test_mask_directory' not in locals():
    print("Error: Required path variables (model_file_path, test_image_directory, test_mask_directory) are not defined.")
else:
    # Build model and load weights
    model = build_model().to(DEVICE)
    # Load model state dictionary, ensuring weights_only=True for safety
    # Handle potential errors during loading
    try:
        model.load_state_dict(torch.load(model_file_path, map_location=DEVICE, weights_only=True))
        model.eval()
        print(f"Successfully loaded model from {model_file_path}")
    except FileNotFoundError:
        print(f"Error: Model file not found at {model_file_path}")
        model = None # Set model to None if loading fails
    except Exception as e:
        print(f"Error loading model: {e}")
        model = None # Set model to None if loading fails


    if model is not None:
        # Get test image paths and corresponding mask paths
        test_img_paths  = sorted(glob(str(Path(test_image_directory) / '*')))
        test_mask_paths = sorted(glob(str(Path(test_mask_directory) / '*'))) # Load ground truth masks too

        if not test_img_paths:
            print(f"Error: No test images found in the specified directory: {test_image_directory}")
        elif len(test_img_paths) != len(test_mask_paths):
             print(f"Warning: Number of test images ({len(test_img_paths)}) and masks ({len(test_mask_paths)}) do not match. Proceeding with visualization using available masks.")
             # Adjust mask paths to only include those that have a corresponding image
             # This is a simple approach; a more robust solution might involve checking filenames
             test_mask_paths = test_mask_paths[:len(test_img_paths)] # Trim mask list if mismatch

        # Create a dataset for the test set
        test_transforms = get_val_aug()
        test_dataset = OralSegDataset(test_img_paths, test_mask_paths, transforms=test_transforms) # Pass mask paths

        # Create a DataLoader for the test set
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=4, pin_memory=True)

        print(f"Found {len(test_img_paths)} test images for visualization and saving predictions.")

        # --- Process and Save Predictions ---
        for i, (x, y_true, img_path) in enumerate(test_loader):


            # ---------- 모델 추론 ----------
            x = x.to(DEVICE)
            logits = model(x)
            y_pred = logits.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)
            y_true = (y_true.squeeze().cpu().numpy().astype(np.uint8)
                      if y_true is not None else None)

            # ---------- 원본 이미지 정보 ----------
            ori_bgr = cv2.imread(img_path[0])
            ori_rgb = cv2.cvtColor(ori_bgr, cv2.COLOR_BGR2RGB)
            H0, W0 = ori_rgb.shape[:2]

            # ---------- ① 스케일·패딩 계산 ----------
            scale   = IMG_SIZE / max(H0, W0)
            new_h   = int(round(H0 * scale))
            new_w   = int(round(W0 * scale))
            pad_y   = (IMG_SIZE - new_h) // 2
            pad_x   = (IMG_SIZE - new_w) // 2

            # ---------- ② 패딩 제거 ----------
            # y_pred 은 (IMG_SIZE, IMG_SIZE)
            mask_crop = y_pred[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

            if y_true is not None:
                mask_true_crop = y_true[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

            # ---------- ③ 원본 크기로 복원 ----------
            mask_pred = cv2.resize(mask_crop, (W0, H0), interpolation=cv2.INTER_NEAREST)
            mask_true = (cv2.resize(mask_true_crop, (W0, H0), interpolation=cv2.INTER_NEAREST)
                        if y_true is not None else None)

            # ---------- ④ 색상 오버레이 & 투명도 ----------
            ALPHA = 0.2            # 0 = 완전 불투명, 1 = 완전 투명

            # ─ 예측 결과 ───────────────────────────────────
            overlay_pred = np.zeros_like(ori_rgb)               # 빈 캔버스
            overlay_pred[mask_pred == 1] = [0, 0, 255]      # TOOTH  → 연한 하늘색
            overlay_pred[mask_pred == 2] = [255, 255, 0]        # ROOT   → 주황
            blended_pred = cv2.addWeighted(ori_rgb, 1-ALPHA,    # 투명도 적용
                                          overlay_pred, ALPHA, 0)

            # ─ GT(정답) ────────────────────────────────────
            overlay_true = np.zeros_like(ori_rgb)
            if mask_true is not None:
                overlay_true[mask_true == 1] = [0, 255, 0]      # TOOTH  → 초록
                overlay_true[mask_true == 2] = [255, 0, 0]      # ROOT   → 빨강
                blended_true = cv2.addWeighted(ori_rgb, 1-ALPHA,
                                              overlay_true, ALPHA, 0)
            else:
                blended_true = ori_rgb        # 마스크가 없으면 원본 그대로



            # ---------- ⑤ 시각화 ----------
            fig, ax = plt.subplots(1, 3, figsize=(18, 6))
            titles  = ['Original', 'Ground‑Truth Overlay', 'Predicted Overlay']
            imgs    = [ori_rgb, blended_true, blended_pred]


            for a, img, t in zip(ax, imgs, titles):
                a.imshow(img); a.set_title(t); a.axis('off')


            plt.tight_layout()

            print(img_path)
            plt.savefig(os.path.join(PREDICTED_MASKS_DIR, os.path.basename(img_path[0])))
            plt.show()
            #visualized_count += 1


    else:
        print("Model could not be loaded. Cannot save or visualize predictions.")

In [ ]:
# prompt: '/content/content/Dataset_split_train_test_266/test/predicted_masks'폴더를 tar 파일로 압축해 줘

!tar -cvf /content/predicted_masks.tar /content/content/Dataset_split_train_test_266/test/predicted_masks

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Ensure necessary variables are available from the previous execution
# metrics_np_arrays, test_filenames, test_root_mask_areas

if 'metrics_np_arrays' not in locals() or 'test_filenames' not in locals() or 'test_root_mask_areas' not in locals():
    print("Please ensure the previous cell calculating test set metrics has been executed.")
else:
    # Get Root Dice scores and corresponding filenames and Root mask areas
    root_dice_scores = metrics_np_arrays['dice']['Root']
    image_filenames = test_filenames
    root_mask_areas = test_root_mask_areas

    # Create a DataFrame for easier handling
    df_outliers = pd.DataFrame({
        'Filename': image_filenames,
        'Root_Dice': root_dice_scores,
        'Root_Mask_Area': root_mask_areas
    })

    # Remove rows with NaN Root_Dice scores (for images where Root class was not present)
    df_outliers = df_outliers.dropna(subset=['Root_Dice'])

    if df_outliers.empty:
        print("No valid Root Dice scores to analyze.")
    else:
        # --- Analyze Outliers based on Root Dice Distribution ---

        # Calculate quartiles and IQR for Root Dice scores
        Q1 = df_outliers['Root_Dice'].quantile(0.25)
        Q3 = df_outliers['Root_Dice'].quantile(0.75)
        IQR = Q3 - Q1

        # Define outlier boundaries
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Identify outlier images (those with Dice scores below the lower bound)
        # We are interested in low performance outliers
        outlier_df = df_outliers[df_outliers['Root_Dice'] < lower_bound].sort_values(by='Root_Dice')

        print("\n--- Outlier Analysis (Root Dice) ---")
        print(f"Q1 (25th percentile): {Q1:.4f}")
        print(f"Q3 (75th percentile): {Q3:.4f}")
        print(f"IQR: {IQR:.4f}")
        print(f"Lower bound for outliers: {lower_bound:.4f}")
        print(f"Upper bound for outliers: {upper_bound:.4f}")

        print(f"\nIdentified {len(outlier_df)} outlier images (Root Dice < {lower_bound:.4f}):")

        if not outlier_df.empty:
            # Sort outliers by Root Dice score (lowest first)
            outlier_df_sorted = outlier_df.sort_values(by='Root_Dice')

            # Display outlier information
            print("\nOutlier Images and their Root Dice Score and Root Mask Area:")
            print(outlier_df_sorted[['Filename', 'Root_Dice', 'Root_Mask_Area']].to_string(index=False))

            # --- Visualize Root Dice Distribution with Outliers ---

            plt.figure(figsize=(10, 6))
            sns.boxplot(y=df_outliers['Root_Dice'])
            plt.title('Boxplot of Root Dice Scores on Test Set')
            plt.ylabel('Root Dice Score')
            plt.show()

            # --- Analyze Root Mask Area Distribution for Outliers vs. Non-Outliers ---

            # Separate non-outliers
            non_outlier_df = df_outliers[df_outliers['Root_Dice'] >= lower_bound]

            # Visualize the distribution of Root Mask Area for outliers and non-outliers
            plt.figure(figsize=(12, 6))
            sns.histplot(data=non_outlier_df, x='Root_Mask_Area', color='skyblue', label='Non-Outliers', kde=True, stat='density', common_norm=False)
            if not outlier_df.empty:
                 sns.histplot(data=outlier_df, x='Root_Mask_Area', color='salmon', label='Outliers', kde=True, stat='density', common_norm=False)
            plt.title('Distribution of Root Mask Area for Outliers vs. Non-Outliers')
            plt.xlabel('Root Mask Area (pixels)')
            plt.ylabel('Density')
            plt.legend()
            plt.show()

            # You can also compare summary statistics
            print("\nSummary Statistics of Root Mask Area:")
            print("Non-Outliers:")
            print(non_outlier_df['Root_Mask_Area'].describe())
            if not outlier_df.empty:
                print("\nOutliers:")
                print(outlier_df['Root_Mask_Area'].describe())

        else:
            print("No outlier images identified based on the Root Dice score lower bound.")

In [ ]:
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
from glob import glob
from pathlib import Path

# Define paths
predicted_masks_dir = '/content/content/Dataset_split_train_test_266/test/predicted_masks'
original_images_dir = '/content/content/Dataset_split_train_test_266/test/images'
# Define directory to save visualizations
VISUALIZATION_SAVE_DIR = '/content/visualizations/'
os.makedirs(VISUALIZATION_SAVE_DIR, exist_ok=True)
print(f"Visualizations will be saved to: {VISUALIZATION_SAVE_DIR}")


# Get a list of predicted mask paths
predicted_mask_paths = sorted(glob(str(Path(predicted_masks_dir) / '*')))

if not predicted_mask_paths:
    print(f"Error: No predicted masks found in the directory: {predicted_masks_dir}")
else:
    print(f"Found {len(predicted_mask_paths)} predicted masks for visualization.")

    # --- Visualize Overlays ---
    num_images_to_visualize = 10  # Number of images to visualize
    visualized_count = 0

    for mask_path in predicted_mask_paths:
        if visualized_count >= num_images_to_visualize:
            break  # Stop after visualizing the desired number of images

        # Get the base filename (without extension) from the mask path
        mask_filename = os.path.basename(mask_path)
        base_filename = os.path.splitext(mask_filename)[0]

        # Construct the path to the corresponding original image
        # Assuming original images have a common extension like .jpg or .png
        # We need to find the original image file with the same base name.
        # A robust way is to glob for files with the base name and any common image extension.
        original_image_patterns = [
            os.path.join(original_images_dir, f"{base_filename}.jpg"),
            os.path.join(original_images_dir, f"{base_filename}.png"),
            os.path.join(original_images_dir, f"{base_filename}.jpeg")
        ]
        original_image_path = None
        for pattern in original_image_patterns:
            if os.path.exists(pattern):
                original_image_path = pattern
                break

        if original_image_path and os.path.exists(mask_path):
            # Load the original image and predicted mask
            original_image = cv2.imread(original_image_path)
            original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
            predicted_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # Ensure image and mask have the same dimensions for overlay
            # Resize the original image to match the mask size
            mask_h, mask_w = predicted_mask.shape
            original_image_resized = cv2.resize(original_image_rgb, (mask_w, mask_h))


            # Create an overlay: colorize the predicted mask and blend it with the original image
            # Example: Colorize Tooth (value 1) in green and Root (value 2) in blue
            overlay = original_image_resized.copy()
            # Check for the class values used in your masks (0 for background, 1 for Tooth, 2 for Root)
            overlay[predicted_mask == 1] = [135, 206, 250]   # TOOTH → 연한 하늘색
            overlay[predicted_mask == 2] = [255, 165, 0]     # ROOT  → 주황 # Changed Root color back to orange


            # Blend the original image and the overlay
            alpha = 0.5  # Transparency factor (0.0 for pure image, 1.0 for pure overlay)
            blended_image = cv2.addWeighted(original_image_resized, 1 - alpha, overlay, alpha, 0)

            # Display the original image and the blended image
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            axes[0].imshow(original_image_resized)
            axes[0].set_title(f'Original Image: {os.path.basename(original_image_path)}')
            axes[0].axis('off')

            axes[1].imshow(blended_image)
            axes[1].set_title('Predicted Mask Overlay')
            axes[1].axis('off')

            plt.tight_layout()

            # Save the figure
            save_filename = f"overlay_{base_filename}.png" # Example filename
            save_path = os.path.join(VISUALIZATION_SAVE_DIR, save_filename)
            plt.savefig(save_path)
            print(f"Saved visualization for {os.path.basename(original_image_path)} to {save_path}")

            plt.close(fig) # Close the figure to free up memory

            #visualized_count += 1
        else:
            print(f"Skipping {mask_filename}: Corresponding original image not found.")

    if visualized_count == 0:
        print("No images were visualized. Please check the paths and filenames.")

In [ ]:
# prompt: '/content/predicted_masks.tar'를 '/content/drive/MyDrive/Colab Notebooks/Root_seg/Test_compare'폴더로 복사해 줘

!cp /content/predicted_masks.tar '/content/drive/MyDrive/Colab Notebooks/Root_seg/Test_compare'

In [ ]:
# prompt: '/content/unetpp_fold1_best.pth'를 이용해서 '/content/Dataset_splited/test'에 대한 성능지표의 95%ci를 계산해 줘

from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score, MulticlassSpecificity
)
import torch
import numpy as np
import pandas as pd
import os
from glob import glob
import cv2
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy import stats # for confidence interval
from pathlib import Path # Import Path here


# Define constants (ensure these match your training constants)
IMG_SIZE    = 736
BATCH_SIZE  = 1 # Use batch size 1 for individual image metrics
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3               # background + Root (0, 1) # Corrected: background + Tooth + Root (0, 1, 2)

# Re-define the OralSegDataset and data augmentation (make sure they are consistent)
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
        if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            aug = self.tf(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        mask = mask.squeeze().long()
        # Return the original image path along with tensors
        return img, mask, self.imgs[idx]


def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# Re-define the model building function
def build_model():
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES,
        in_channels    = 3,
        activation     = None
    )

# Function to calculate Dice score manually
def calculate_dice(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

# Function to calculate IoU manually
def calculate_iou(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

# Function to calculate F1 score manually
def calculate_f1(pred, target, smooth=1e-6):
    tp = np.sum((pred == 1) & (target == 1))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))

    # Handle division by zero for precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Handle division by zero for F1 score
    f1 = (2. * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1

def calculate_ap(pred, target, iou_threshold=0.5):
    """
    Calculate Average Precision (AP) for a single class.
    """
    iou = calculate_iou(pred, target)
    if iou >= iou_threshold:
        return 1.0
    else:
        return 0.0


# Function to calculate confidence interval
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    if n <= 1:
        # Cannot compute CI with 1 or fewer samples
        return np.nan, np.nan, np.nan
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

# --- Evaluation on Test Set ---
def evaluate_test_set_and_collect_metrics(model_path, test_image_dir, test_mask_dir):
    """
    Loads a trained model, performs inference on the test set,
    and collects per-image metrics (Dice, IoU, Precision, Recall, Specificity, Accuracy, F1)
    into numpy arrays. Also computes 95% CI. Returns image filenames and Root mask areas.

    Args:
        model_path (str): Path to the trained model file (.pth).
        test_image_dir (str): Directory containing the test images.
        test_mask_dir (str): Directory containing the ground truth test masks.

    Returns:
        tuple: A tuple containing:
            - dict: A dictionary containing numpy arrays for each metric (Dice, IoU, etc.)
                    for each class ('Overall', 'Background', 'Tooth', 'Root').
            - dict: A dictionary for mean metrics.
            - dict: A dictionary for 95% CI for each metric/class.
            - list: A list of processed image filenames.
            - list: A list of corresponding 'Tooth' mask areas.
            - list: A list of corresponding 'Root' mask areas.
    """
    print("\n--- Evaluating on Test Set and Collecting Metrics ---")

    # Build model and load weights
    model = build_model().to(DEVICE)
    # Fix: Add weights_only=True to torch.load for safety
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    # Get test image and mask paths
    test_img_paths  = sorted(glob(str(Path(test_image_dir) / '*')))
    test_mask_paths = sorted(glob(str(Path(test_mask_dir) / '*')))

    if not test_img_paths or not test_mask_paths:
        print("Error: No test images or masks found in the specified directories.")
        return {}, {}, {}, [], [], [] # Return empty dictionaries and lists, adjusted for two mask areas

    assert len(test_img_paths) == len(test_mask_paths), "Test images and masks count mismatch."
    print(f"Found {len(test_img_paths)} test images and masks.")

    # Create a dataset for the test set
    test_transforms = get_val_aug()
    # Ensure OralSegDataset returns image path, tooth_mask_area, root_mask_area
    # Note: Calculating mask area here might be slow if done on the fly.
    # It might be better to calculate areas separately or load pre-calculated areas.
    # For simplicity in this fix, let's assume mask area calculation is handled efficiently by the dataset or we accept the potential overhead.
    test_dataset = OralSegDataset(test_img_paths, test_mask_paths, transforms=test_transforms)

    # Create a DataLoader for the test set
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)

    # Initialize lists to store per-image metrics and additional info
    all_metrics = {
        'dice': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'iou': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'prec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'rec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'spec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'acc': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'f1': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added F1 score
        'mAP50': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added mAP50 score
    }
    processed_filenames = []
    tooth_mask_areas = []
    root_mask_areas = []


    # --- Inference and Evaluation ---
    with torch.no_grad():
        # Unpack image, mask, and image path from the DataLoader
        for i, (x, y, img_path) in enumerate(test_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Store filename (get filename from path)
            processed_filenames.append(os.path.basename(img_path[0])) # Batch size is 1

            # Calculate Tooth and Root mask areas (on CPU)
            tooth_area = (y.cpu() == 1).sum().item()
            root_area = (y.cpu() == 2).sum().item() # Root is now class 2
            tooth_mask_areas.append(tooth_area)
            root_mask_areas.append(root_area)


            # Perform inference
            logits = model(x)
            preds = logits.argmax(dim=1) # Get predicted class index

            # Move predicted and target tensors to CPU for NumPy calculations
            preds_np = preds.cpu().numpy()
            y_np = y.cpu().numpy()

            dice_scores = []
            iou_scores = []
            acc_scores_np = []
            prec_scores_np = []
            rec_scores_np = []
            spec_scores_np = []
            f1_scores_np = [] # Added F1 score list
            ap50_scores_np = [] # Added AP50 score list

            # Calculate Dice, IoU, Accuracy, Precision, Recall, Specificity, F1 for each class using numpy
            for cls_id in range(N_CLASSES):
                pred_cls = (preds_np == cls_id).astype(np.uint8)
                target_cls = (y_np == cls_id).astype(np.uint8)

                # Calculate Dice and IoU (manual calculation is fine)
                # Avoid division by zero if a class is not present in the ground truth or prediction for this image
                if target_cls.sum() > 0 or pred_cls.sum() > 0:
                     dice = calculate_dice(pred_cls, target_cls)
                     iou = calculate_iou(pred_cls, target_cls)
                     dice_scores.append(dice)
                     iou_scores.append(iou)
                else:
                     # If neither predicted nor ground truth pixels for this class, metric is typically 1.0 (perfect score as there's nothing to segment)
                     # Append 1.0 as per convention for empty ground truth/prediction
                     dice_scores.append(1.0)
                     iou_scores.append(1.0)


                # Calculate Accuracy, Precision, Recall, Specificity, F1 using numpy
                tp = np.sum((pred_cls == 1) & (target_cls == 1))
                fp = np.sum((pred_cls == 1) & (target_cls == 0))
                tn = np.sum((pred_cls == 0) & (target_cls == 0))
                fn = np.sum((pred_cls == 0) & (target_cls == 1))

                # Calculate metrics (handle division by zero) - append NaN if denominator is zero
                accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan
                precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
                recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan # Also known as Sensitivity
                specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
                f1 = calculate_f1(pred_cls, target_cls) # Calculate F1 using the new function
                ap50 = calculate_ap(pred_cls, target_cls)

                acc_scores_np.append(accuracy)
                prec_scores_np.append(precision)
                rec_scores_np.append(recall)
                spec_scores_np.append(specificity)
                f1_scores_np.append(f1) # Added F1 score
                ap50_scores_np.append(ap50) # Added AP50 score


            # Store per-class metrics in the dictionary
            class_indices = {0: 'Background', 1: 'Tooth', 2: 'Root'} # Updated class indices
            metric_values = {
                'dice': dice_scores,
                'iou': iou_scores,
                'prec': prec_scores_np,
                'rec': rec_scores_np,
                'spec': spec_scores_np,
                'acc': acc_scores_np,
                'f1': f1_scores_np, # Added F1 score
                'mAP50': ap50_scores_np,
            }

            for metric_name, values in metric_values.items():
                for cls_id, cls_name in class_indices.items():
                    if cls_id < len(values):
                         all_metrics[metric_name][cls_name].append(values[cls_id])
                    else:
                         # This case should ideally not happen if values list has N_CLASSES elements
                         all_metrics[metric_name][cls_name].append(np.nan)


            # Calculate "Overall" metrics (mean over Tooth and Root) for this image
            # Append to the Overall list - use nanmean to handle NaNs from classes not present
            # Note: If both Tooth and Root are not present and metrics are 1.0, the mean will be 1.0.
            # If one is present and the other not (1.0), mean is average. If both NaN, mean is NaN.
            all_metrics['dice']['Overall'].append(np.nanmean([dice_scores[1], dice_scores[2]])) # mean of Tooth and Root Dice
            all_metrics['iou']['Overall'].append(np.nanmean([iou_scores[1], iou_scores[2]])) # mean of Tooth and Root IoU
            all_metrics['prec']['Overall'].append(np.nanmean([prec_scores_np[1], prec_scores_np[2]])) # mean of Tooth and Root Precision
            all_metrics['rec']['Overall'].append(np.nanmean([rec_scores_np[1], rec_scores_np[2]])) # mean of Tooth and Root Recall
            all_metrics['spec']['Overall'].append(np.nanmean([spec_scores_np[1], spec_scores_np[2]])) # mean of Tooth and Root Specificity
            # Overall pixel accuracy for the whole image - this is typically the Background accuracy when average=None
            all_metrics['acc']['Overall'].append(acc_scores_np[0]) # Background pixel accuracy as overall accuracy
            all_metrics['f1']['Overall'].append(np.nanmean([f1_scores_np[1], f1_scores_np[2]])) # mean of Tooth and Root F1
            all_metrics['mAP50']['Overall'].append(np.nanmean([ap50_scores_np[1], ap50_scores_np[2]]))


            print(f"Processed test image {i+1}/{len(test_img_paths)}")


    # Convert lists to numpy arrays
    metrics_np_arrays = {}
    for metric_name, classes_data in all_metrics.items():
        metrics_np_arrays[metric_name] = {}
        for cls_name, values_list in classes_data.items():
            metrics_np_arrays[metric_name][cls_name] = np.array(values_list)

    # Calculate Mean and 95% Confidence Interval
    mean_metrics = {}
    ci_metrics = {} # Stores (mean, lower_bound, upper_bound)
    for metric_name, classes_data in metrics_np_arrays.items():
        mean_metrics[metric_name] = {}
        ci_metrics[metric_name] = {}
        for cls_name, values_array in classes_data.items():
            # Calculate mean ignoring NaNs
            mean_metrics[metric_name][cls_name] = np.nanmean(values_array)
            # Calculate CI ignoring NaNs - pass only non-NaN values to mean_confidence_interval
            ci_metrics[metric_name][cls_name] = mean_confidence_interval(values_array[~np.isnan(values_array)])


    print("\n--- Test Set Evaluation Summary ---")
    print("Metrics (Mean ± 95% CI):")
    class_order = ['Overall', 'Background', 'Tooth', 'Root'] # Updated order for printing

    for cls_name in class_order:
         print(f"\nClass: {cls_name}")
         for metric_name in all_metrics.keys():
             mean_val = mean_metrics[metric_name][cls_name]
             ci_info = ci_metrics[metric_name][cls_name]

             # Format CI string (handle cases with not enough data for CI)
             if len(ci_info) == 3 and not np.isnan(ci_info[1]) and not np.isnan(ci_info[2]):
                 ci_str = f" (CI: {ci_info[1]:.4f} - {ci_info[2]:.4f})"
             else:
                 ci_str = " (CI: N/A)"

             print(f"  {metric_name.upper():<5s}: {mean_val:.4f}{ci_str}")

    # --- Save Mean and CI to CSV ---
    output_data = []
    for metric_name in all_metrics.keys():
        for cls_name in class_order:
            mean_val = mean_metrics[metric_name][cls_name]
            ci_info = ci_metrics[metric_name][cls_name]
            output_data.append({
                'Class': cls_name,
                'Metric': metric_name,
                'Mean': mean_val,
                'CI_Lower': ci_info[1] if len(ci_info) == 3 else np.nan,
                'CI_Upper': ci_info[2] if len(ci_info) == 3 else np.nan
            })

    df_mean_ci = pd.DataFrame(output_data)
    output_csv_path = '/content/test_set_mean_ci_metrics.csv'
    df_mean_ci.to_csv(output_csv_path, index=False)
    print(f"\nMean and CI results saved to {output_csv_path}")


    # Return Tooth and Root mask areas and filenames
    return metrics_np_arrays, mean_metrics, ci_metrics, processed_filenames, tooth_mask_areas, root_mask_areas


# Define paths for the test set evaluation
# Use fold 5 model as it is likely the best based on the training logs
model_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth' # Use the fold 5 best model
test_image_directory = '/content/content/Dataset_split_train_test_266/test/images'
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks'

# Run the evaluation and get the results as numpy arrays and CI, filenames, and mask areas
metrics_np_arrays, mean_metrics, ci_metrics, test_filenames, test_tooth_mask_areas, test_root_mask_areas = evaluate_test_set_and_collect_metrics(
    model_file_path, test_image_directory, test_mask_directory
)

# Now you have the metrics stored in numpy arrays in `metrics_np_arrays`,
# the mean and CI in `mean_metrics` and `ci_metrics`,
# the processed filenames in `test_filenames`, and the root mask areas in `test_root_mask_areas`.
# For example, the Dice scores for the Root class for each image are in:
# metrics_np_arrays['dice']['Root']

# The corresponding filenames are in `test_filenames` and root mask areas in `test_root_mask_areas`
# You can use these lists to analyze outlier cases based on their filenames and mask areas.

In [ ]:
# prompt: '/content/unetpp_fold1_best.pth'를 이용해서 '/content/Dataset_splited/test'에 대한 성능지표의 95%ci를 계산해 줘

from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score, MulticlassSpecificity
)
import torch
import numpy as np
import pandas as pd
import os
from glob import glob
import cv2
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy import stats # for confidence interval
from pathlib import Path # Import Path here


# Define constants (ensure these match your training constants)
IMG_SIZE    = 736
BATCH_SIZE  = 1 # Use batch size 1 for individual image metrics
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3               # background + Root (0, 1) # Corrected: background + Tooth + Root (0, 1, 2)

# Re-define the OralSegDataset and data augmentation (make sure they are consistent)
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
        if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            aug = self.tf(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        mask = mask.squeeze().long()
        # Return the original image path along with tensors
        return img, mask, self.imgs[idx]


def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# Re-define the model building function
def build_model():
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES,
        in_channels    = 3,
        activation     = None
    )

# Function to calculate Dice score manually
def calculate_dice(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

# Function to calculate IoU manually
def calculate_iou(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

# Function to calculate F1 score manually
def calculate_f1(pred, target, smooth=1e-6):
    tp = np.sum((pred == 1) & (target == 1))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))

    # Handle division by zero for precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Handle division by zero for F1 score
    f1 = (2. * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1

def calculate_ap(pred, target, iou_threshold=0.5):
    """
    Calculate Average Precision (AP) for a single class.
    """
    iou = calculate_iou(pred, target)
    if iou >= iou_threshold:
        return 1.0
    else:
        return 0.0


# Function to calculate confidence interval using bootstrapping
def mean_confidence_interval(data, confidence=0.95, n_bootstraps=1000):
    a = 1.0 * np.array(data)
    n = len(a)
    if n == 0: # Handle empty data case
        return np.nan, np.nan, np.nan
    if n == 1: # Cannot compute CI with 1 sample, return mean and NaN for bounds
        return np.mean(a), np.nan, np.nan

    # Resample with replacement and calculate mean for each bootstrap sample
    bootstrap_means = []
    # Use a fixed random seed for reproducibility if needed, or remove for true randomness
    # np.random.seed(42)
    for _ in range(n_bootstraps):
        bootstrap_sample = np.random.choice(a, size=n, replace=True)
        # Calculate mean, ignoring NaNs if present in the bootstrap sample
        bootstrap_means.append(np.nanmean(bootstrap_sample))

    # Remove NaN means from bootstrap samples (can happen if all samples in a bootstrap are NaN)
    bootstrap_means = np.array(bootstrap_means)
    bootstrap_means = bootstrap_means[~np.isnan(bootstrap_means)]

    if len(bootstrap_means) == 0: # If all bootstrap samples resulted in NaN mean
        return np.nan, np.nan, np.nan

    # Calculate the confidence interval from the distribution of bootstrap means
    # Use percentiles to get the confidence interval
    lower_bound = np.percentile(bootstrap_means, (1 - confidence) / 2 * 100)
    upper_bound = np.percentile(bootstrap_means, (1 + confidence) / 2 * 100)
    mean_val = np.mean(bootstrap_means) # Or np.mean(a) for the original mean

    return mean_val, lower_bound, upper_bound


# --- Evaluation on Test Set ---
def evaluate_test_set_and_collect_metrics(model_path, test_image_dir, test_mask_dir):
    """
    Loads a trained model, performs inference on the test set,
    and collects per-image metrics (Dice, IoU, Precision, Recall, Specificity, Accuracy, F1)
    into numpy arrays. Also computes 95% CI using bootstrapping. Returns image filenames and Root mask areas.

    Args:
        model_path (str): Path to the trained model file (.pth).
        test_image_dir (str): Directory containing the test images.
        test_mask_dir (str): Directory containing the ground truth test masks.

    Returns:
        tuple: A tuple containing:
            - dict: A dictionary containing numpy arrays for each metric (Dice, IoU, etc.)
                    for each class ('Overall', 'Background', 'Tooth', 'Root').
            - dict: A dictionary for mean metrics.
            - dict: A dictionary for 95% CI for each metric/class.
            - list: A list of processed image filenames.
            - list: A list of corresponding 'Tooth' mask areas.
            - list: A list of corresponding 'Root' mask areas.
    """
    print("\n--- Evaluating on Test Set and Collecting Metrics ---")

    # Build model and load weights
    model = build_model().to(DEVICE)
    # Fix: Add weights_only=True to torch.load for safety
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    # Get test image and mask paths
    test_img_paths  = sorted(glob(str(Path(test_image_dir) / '*')))
    test_mask_paths = sorted(glob(str(Path(test_mask_dir) / '*')))

    if not test_img_paths or not test_mask_paths:
        print("Error: No test images or masks found in the specified directories.")
        return {}, {}, {}, [], [], [] # Return empty dictionaries and lists, adjusted for two mask areas

    assert len(test_img_paths) == len(test_mask_paths), "Test images and masks count mismatch."
    print(f"Found {len(test_img_paths)} test images and masks.")

    # Create a dataset for the test set
    test_transforms = get_val_aug()
    # Ensure OralSegDataset returns image path, tooth_mask_area, root_mask_area
    # Note: Calculating mask area here might be slow if done on the fly.
    # It might be better to calculate areas separately or load pre-calculated areas.
    # For simplicity in this fix, let's assume mask area calculation is handled efficiently by the dataset or we accept the potential overhead.
    test_dataset = OralSegDataset(test_img_paths, test_mask_paths, transforms=test_transforms)

    # Create a DataLoader for the test set
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)

    # Initialize lists to store per-image metrics and additional info
    all_metrics = {
        'dice': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'iou': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'prec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'rec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'spec': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'acc': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []},
        'f1': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added F1 score
        'mAP50': {'Overall': [], 'Background': [], 'Tooth': [], 'Root': []}, # Added mAP50 score
    }
    processed_filenames = []
    tooth_mask_areas = []
    root_mask_areas = []


    # --- Inference and Evaluation ---
    with torch.no_grad():
        # Unpack image, mask, and image path from the DataLoader
        for i, (x, y, img_path) in enumerate(test_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Store filename (get filename from path)
            processed_filenames.append(os.path.basename(img_path[0])) # Batch size is 1

            # Calculate Tooth and Root mask areas (on CPU)
            tooth_area = (y.cpu() == 1).sum().item()
            root_area = (y.cpu() == 2).sum().item() # Root is now class 2
            tooth_mask_areas.append(tooth_area)
            root_mask_areas.append(root_area)


            # Perform inference
            logits = model(x)
            preds = logits.argmax(dim=1) # Get predicted class index

            # Move predicted and target tensors to CPU for NumPy calculations
            preds_np = preds.cpu().numpy()
            y_np = y.cpu().numpy()

            dice_scores = []
            iou_scores = []
            acc_scores_np = []
            prec_scores_np = []
            rec_scores_np = []
            spec_scores_np = []
            f1_scores_np = [] # Added F1 score list
            ap50_scores_np = [] # Added AP50 score list

            # Calculate Dice, IoU, Accuracy, Precision, Recall, Specificity, F1 for each class using numpy
            for cls_id in range(N_CLASSES):
                pred_cls = (preds_np == cls_id).astype(np.uint8)
                target_cls = (y_np == cls_id).astype(np.uint8)

                # Calculate Dice and IoU (manual calculation is fine)
                # Avoid division by zero if a class is not present in the ground truth or prediction for this image
                if target_cls.sum() > 0 or pred_cls.sum() > 0:
                     dice = calculate_dice(pred_cls, target_cls)
                     iou = calculate_iou(pred_cls, target_cls)
                     dice_scores.append(dice)
                     iou_scores.append(iou)
                else:
                     # If neither predicted nor ground truth pixels for this class, metric is typically 1.0 (perfect score as there's nothing to segment)
                     # Append 1.0 as per convention for empty ground truth/prediction
                     dice_scores.append(1.0)
                     iou_scores.append(1.0)


                # Calculate Accuracy, Precision, Recall, Specificity, F1 using numpy
                tp = np.sum((pred_cls == 1) & (target_cls == 1))
                fp = np.sum((pred_cls == 1) & (target_cls == 0))
                tn = np.sum((pred_cls == 0) & (target_cls == 0))
                fn = np.sum((pred_cls == 0) & (target_cls == 1))

                # Calculate metrics (handle division by zero) - append NaN if denominator is zero
                accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan
                precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
                recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan # Also known as Sensitivity
                specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
                f1 = calculate_f1(pred_cls, target_cls) # Calculate F1 using the new function
                ap50 = calculate_ap(pred_cls, target_cls)

                acc_scores_np.append(accuracy)
                prec_scores_np.append(precision)
                rec_scores_np.append(recall)
                spec_scores_np.append(specificity)
                f1_scores_np.append(f1) # Added F1 score
                ap50_scores_np.append(ap50) # Added AP50 score


            # Store per-class metrics in the dictionary
            class_indices = {0: 'Background', 1: 'Tooth', 2: 'Root'} # Updated class indices
            metric_values = {
                'dice': dice_scores,
                'iou': iou_scores,
                'prec': prec_scores_np,
                'rec': rec_scores_np,
                'spec': spec_scores_np,
                'acc': acc_scores_np,
                'f1': f1_scores_np, # Added F1 score
                'mAP50': ap50_scores_np,
            }

            for metric_name, values in metric_values.items():
                for cls_id, cls_name in class_indices.items():
                    if cls_id < len(values):
                         all_metrics[metric_name][cls_name].append(values[cls_id])
                    else:
                         # This case should ideally not happen if values list has N_CLASSES elements
                         all_metrics[metric_name][cls_name].append(np.nan)


            # Calculate "Overall" metrics (mean over Tooth and Root) for this image
            # Append to the Overall list - use nanmean to handle NaNs from classes not present
            # Note: If both Tooth and Root are not present and metrics are 1.0, the mean will be 1.0.
            # If one is present and the other not (1.0), mean is average. If both NaN, mean is NaN.
            all_metrics['dice']['Overall'].append(np.nanmean([dice_scores[1], dice_scores[2]])) # mean of Tooth and Root Dice
            all_metrics['iou']['Overall'].append(np.nanmean([iou_scores[1], iou_scores[2]])) # mean of Tooth and Root IoU
            all_metrics['prec']['Overall'].append(np.nanmean([prec_scores_np[1], prec_scores_np[2]])) # mean of Tooth and Root Precision
            all_metrics['rec']['Overall'].append(np.nanmean([rec_scores_np[1], rec_scores_np[2]])) # mean of Tooth and Root Recall
            all_metrics['spec']['Overall'].append(np.nanmean([spec_scores_np[1], spec_scores_np[2]])) # mean of Tooth and Root Specificity
            # Overall pixel accuracy for the whole image - this is typically the Background accuracy when average=None
            all_metrics['acc']['Overall'].append(acc_scores_np[0]) # Background pixel accuracy as overall accuracy
            all_metrics['f1']['Overall'].append(np.nanmean([f1_scores_np[1], f1_scores_np[2]])) # mean of Tooth and Root F1
            all_metrics['mAP50']['Overall'].append(np.nanmean([ap50_scores_np[1], ap50_scores_np[2]]))


            print(f"Processed test image {i+1}/{len(test_img_paths)}")


    # Convert lists to numpy arrays
    metrics_np_arrays = {}
    for metric_name, classes_data in all_metrics.items():
        metrics_np_arrays[metric_name] = {}
        for cls_name, values_list in classes_data.items():
            metrics_np_arrays[metric_name][cls_name] = np.array(values_list)

    # Calculate Mean and 95% Confidence Interval using the modified function
    mean_metrics = {}
    ci_metrics = {} # Stores (mean, lower_bound, upper_bound)
    for metric_name, classes_data in metrics_np_arrays.items():
        mean_metrics[metric_name] = {}
        ci_metrics[metric_name] = {}
        for cls_name, values_array in classes_data.items():
            # Calculate mean ignoring NaNs
            mean_metrics[metric_name][cls_name] = np.nanmean(values_array)
            # Calculate CI ignoring NaNs using bootstrapping
            ci_metrics[metric_name][cls_name] = mean_confidence_interval(values_array[~np.isnan(values_array)], n_bootstraps=1000)


    print("\n--- Test Set Evaluation Summary (Bootstrapped CI) ---")
    print("Metrics (Mean ± 95% CI):")
    class_order = ['Overall', 'Background', 'Tooth', 'Root'] # Updated order for printing

    for cls_name in class_order:
         print(f"\nClass: {cls_name}")
         for metric_name in all_metrics.keys():
             mean_val = mean_metrics[metric_name][cls_name]
             ci_info = ci_metrics[metric_name][cls_name]

             # Format CI string (handle cases with not enough data for CI)
             if len(ci_info) == 3 and not np.isnan(ci_info[1]) and not np.isnan(ci_info[2]):
                 ci_str = f" (CI: {ci_info[1]:.4f} - {ci_info[2]:.4f})"
             else:
                 ci_str = " (CI: N/A)"

             print(f"  {metric_name.upper():<5s}: {mean_val:.4f}{ci_str}")

    # --- Save Mean and CI to CSV ---
    output_data = []
    for metric_name in all_metrics.keys():
        for cls_name in class_order:
            mean_val = mean_metrics[metric_name][cls_name]
            ci_info = ci_metrics[metric_name][cls_name]
            output_data.append({
                'Class': cls_name,
                'Metric': metric_name,
                'Mean': mean_val,
                'CI_Lower': ci_info[1] if len(ci_info) == 3 else np.nan,
                'CI_Upper': ci_info[2] if len(ci_info) == 3 else np.nan
            })

    df_mean_ci = pd.DataFrame(output_data)
    output_csv_path = '/content/test_set_mean_ci_metrics_bootstrapped.csv' # Save to a new file
    df_mean_ci.to_csv(output_csv_path, index=False)
    print(f"\nMean and CI results saved to {output_csv_path}")


    # Return Tooth and Root mask areas and filenames
    return metrics_np_arrays, mean_metrics, ci_metrics, processed_filenames, tooth_mask_areas, root_mask_areas


# Define paths for the test set evaluation
# Use fold 5 model as it is likely the best based on the training logs
model_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth' # Use the fold 5 best model
test_image_directory = '/content/content/Dataset_split_train_test_266/test/images'
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks'

# Run the evaluation and get the results as numpy arrays and CI, filenames, and mask areas
metrics_np_arrays, mean_metrics, ci_metrics, test_filenames, test_tooth_mask_areas, test_root_mask_areas = evaluate_test_set_and_collect_metrics(
    model_file_path, test_image_directory, test_mask_directory
)

# Now you have the metrics stored in numpy arrays in `metrics_np_arrays`,
# the mean and CI in `mean_metrics` and `ci_metrics`,
# the processed filenames in `test_filenames`, and the root mask areas in `test_root_mask_areas`.
# For example, the Dice scores for the Root class for each image are in:
# metrics_np_arrays['dice']['Root']

# The corresponding filenames are in `test_filenames` and root mask areas in `test_root_mask_areas`
# You can use these lists to analyze outlier cases based on their filenames and mask areas.

In [ ]:
import matplotlib.pyplot as plt
import os
import cv2
import torch
import torch.nn.functional as F # Import torch.nn.functional
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pathlib import Path
import numpy as np
from glob import glob # Import glob here

# Assuming constants and model building functions (IMG_SIZE, BATCH_SIZE, DEVICE, ENCODER, ENC_WEIGHTS, N_CLASSES,
# OralSegDataset, get_val_aug, build_model) are defined in previous cells and are available in the current runtime.
# If they are not, you would need to redefine them here or ensure the user runs the preceding cells.

# Define constants (ensure these match your training constants)
IMG_SIZE    = 768 # Changed IMG_SIZE for test inference
BATCH_SIZE  = 1 # Use batch size 1 for visualization
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER     = 'resnet34'
ENC_WEIGHTS = 'imagenet'
N_CLASSES   = 3 # background + Tooth + Root

# Re-define the OralSegDataset and data augmentation (make sure they are consistent)
class OralSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, transforms=None):
        self.imgs  = img_paths
        self.masks = mask_paths # masks can be None for prediction
        self.tf    = transforms

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)

        mask = None
        if self.masks:
             mask = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
             if mask.ndim == 2: mask = mask[..., None]

        if self.tf:
            # Apply transformations to image and mask (if mask exists)
            if mask is not None:
                aug = self.tf(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
                mask = mask.squeeze().long() # Ensure mask is squeezed and long type
            else:
                 # Apply transformations only to image
                 aug = self.tf(image=img)
                 img = aug['image']


        return img, mask, self.imgs[idx] # Return image, mask (can be None), and image path


def get_val_aug():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# Re-define the model building function
def build_model():
    return smp.UnetPlusPlus(
        encoder_name   = ENCODER,
        encoder_weights= ENC_WEIGHTS,
        classes        = N_CLASSES,
        in_channels    = 3,
        activation     = None # No activation for multiclass output before loss/argmax
    )


# Define paths for the test set evaluation and saving predictions
model_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/UnetPP_0725_Result_best/unetpp_fold5_best_best.pth' # Use the fold 5 best model
test_image_directory = '/content/content/Dataset_split_train_test_266/test/images'
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks' # Include masks for visualization if needed
PREDICTED_MASKS_DIR = '/content/content/Dataset_split_train_test_266/test/predicted_masks'
os.makedirs(PREDICTED_MASKS_DIR, exist_ok=True)
print(f"Predicted masks will be saved to: {PREDICTED_MASKS_DIR}")


# Check if required variables are defined
if 'model_file_path' not in locals() or 'test_image_directory' not in locals() or 'test_mask_directory' not in locals():
    print("Error: Required path variables (model_file_path, test_image_directory, test_mask_directory) are not defined.")
else:
    # Build model and load weights
    model = build_model().to(DEVICE)
    # Load model state dictionary, ensuring weights_only=True for safety
    # Handle potential errors during loading
    try:
        model.load_state_dict(torch.load(model_file_path, map_location=DEVICE, weights_only=True))
        model.eval()
        print(f"Successfully loaded model from {model_file_path}")
    except FileNotFoundError:
        print(f"Error: Model file not found at {model_file_path}")
        model = None # Set model to None if loading fails
    except Exception as e:
        print(f"Error loading model: {e}")
        model = None # Set model to None if loading fails


    if model is not None:
        # Get test image paths and corresponding mask paths
        test_img_paths  = sorted(glob(str(Path(test_image_directory) / '*')))
        test_mask_paths = sorted(glob(str(Path(test_mask_directory) / '*'))) # Load ground truth masks too

        if not test_img_paths:
            print(f"Error: No test images found in the specified directory: {test_image_directory}")
        elif len(test_img_paths) != len(test_mask_paths):
             print(f"Warning: Number of test images ({len(test_img_paths)}) and masks ({len(test_mask_paths)}) do not match. Proceeding with prediction and saving, but visualization might be affected.")
             # Adjust mask paths to only include those that have a corresponding image
             # This is a simple approach; a more robust solution might involve checking filenames
             test_mask_paths = test_mask_paths[:len(test_img_paths)] # Trim mask list if mismatch


        # Create a dataset for the test set (masks are included for optional visualization later)
        test_transforms = get_val_aug()
        test_dataset = OralSegDataset(test_img_paths, test_mask_paths, transforms=test_transforms)

        # Create a DataLoader for the test set
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=4, pin_memory=True)

        print(f"Found {len(test_img_paths)} test images for prediction and saving.")

        # --- Perform Prediction and Save Masks ---
        with torch.no_grad():
            # Unpack image, mask, and image path from the DataLoader
            for i, (x, y_true, img_path) in enumerate(test_loader):
                x = x.to(DEVICE)

                # Perform inference
                logits = model(x)
                y_pred = logits.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8) # Get predicted class index as numpy array

                # ---------- 원본 이미지 정보 ----------
                ori_bgr = cv2.imread(img_path[0])
                #ori_rgb = cv2.cvtColor(ori_bgr, cv2.COLOR_BGR2RGB) # Not needed for saving mask
                H0, W0 = ori_bgr.shape[:2]

                # ---------- ① 스케일·패딩 계산 ----------
                scale   = IMG_SIZE / max(H0, W0)
                new_h   = int(round(H0 * scale))
                new_w   = int(round(W0 * scale))
                pad_y   = (IMG_SIZE - new_h) // 2
                pad_x   = (IMG_SIZE - new_w) // 2

                # ---------- ② 패딩 제거 ----------
                # y_pred 은 (IMG_SIZE, IMG_SIZE)
                mask_crop = y_pred[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

                # ---------- ③ 원본 크기로 복원 ----------
                mask_pred = cv2.resize(mask_crop, (W0, H0), interpolation=cv2.INTER_NEAREST)


                # ---------- Save the predicted mask ----------
                # Get the original image filename
                original_filename = os.path.basename(img_path[0])
                # Create the predicted mask filename (e.g., same name with .png extension)
                predicted_mask_filename = os.path.splitext(original_filename)[0] + '.png'
                predicted_mask_path = os.path.join(PREDICTED_MASKS_DIR, predicted_mask_filename)

                # Save the predicted mask (grayscale)
                cv2.imwrite(predicted_mask_path, mask_pred)

                print(f"Processed and saved predicted mask for {original_filename} to {predicted_mask_path}")

    else:
        print("Model could not be loaded. Cannot perform prediction and saving.")

Test 이미지에 대해 predictopn mask 생성

In [ ]:
import matplotlib.pyplot as plt
import os
import cv2
import random

# Directory containing the predicted masks
predicted_masks_dir = '/content/content/Dataset_split_train_test_266/test/predicted_masks'

# Get a list of predicted mask filenames
predicted_mask_filenames = os.listdir(predicted_masks_dir)

# Filter out non-PNG files if necessary (assuming masks are .png)
predicted_mask_filenames = [f for f in predicted_mask_filenames if f.endswith('.png')]

# Check if there are enough files to sample
num_images_to_visualize = 10
if len(predicted_mask_filenames) < num_images_to_visualize:
    print(f"Found only {len(predicted_mask_filenames)} predicted mask files. Visualizing all of them.")
    sample_mask_filenames = predicted_mask_filenames
else:
    # Randomly sample num_images_to_visualize mask filenames
    sample_mask_filenames = random.sample(predicted_mask_filenames, num_images_to_visualize)

for mask_filename in sample_mask_filenames:
    # Construct the full path for the mask
    mask_path = os.path.join(predicted_masks_dir, mask_filename)

    # Read the mask
    # Use cv2.IMREAD_UNCHANGED to read as is, then ensure it's grayscale if it's not already
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    # If the mask is not grayscale (e.g., read as BGR), convert it to grayscale
    if len(mask.shape) == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    # Display the mask
    plt.figure(figsize=(6, 6))
    plt.imshow(mask, cmap='gray', vmin=0, vmax=N_CLASSES-1) # Display mask in grayscale with appropriate value range
    plt.title(f'Predicted Mask: {mask_filename}')
    plt.axis('off')
    plt.colorbar(ticks=range(N_CLASSES)) # Add a colorbar to show class values
    plt.show()

In [ ]:
# prompt: '/content/content/Dataset_split_train_test_266/test/predicted_masks'폴더를 tar 파일로 압축해 줘

!tar -cvf /content/predicted_masks.tar /content/content/Dataset_split_train_test_266/test/predicted_masks

In [ ]:
# prompt: '/content/predicted_masks.tar'를 '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare_0729_best'폴더로 복사해 줘

import os

# Define the destination directory in Google Drive
destination_dir = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare_0729_best'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Copy the tar file to the destination directory
!cp /content/predicted_masks.tar "{destination_dir}"

In [ ]:
# prompt: '/content/content/Dataset_split_train_test_266/test/masks'를 이용해 imageid 별로 'Tooth'와 'Root'에 면적을 계산해서 파일로 저장해

import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# Directory containing the test masks
test_mask_directory = '/content/content/Dataset_split_train_test_266/test/masks'

# Get a list of mask file paths
mask_paths = sorted(glob(os.path.join(test_mask_directory, '*.png')))

if not mask_paths:
    print(f"Error: No mask files found in the directory: {test_mask_directory}")
else:
    print(f"Found {len(mask_paths)} mask files.")

    # List to store results
    mask_area_data = []

    # Iterate through each mask file
    for mask_path in mask_paths:
        # Get the filename (which can be used as image ID)
        mask_filename = os.path.basename(mask_path)
        image_id = os.path.splitext(mask_filename)[0] # Use filename without extension as ID

        # Read the mask
        # Use cv2.IMREAD_UNCHANGED to read as is, then ensure it's grayscale
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

        # If the mask is not grayscale, convert it
        if len(mask.shape) == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        # Ensure mask values are compatible with our class IDs (0: background, 1: Tooth, 2: Root)
        # If mask values are different, you might need to remap them here.
        # Assuming mask values correspond to class IDs.

        # Calculate area for each class (number of pixels with that class ID)
        # Assuming class ID 1 is 'Tooth' and class ID 2 is 'Root'
        tooth_area = np.sum(mask == 1)
        root_area = np.sum(mask == 2)

        # Append data to the list
        mask_area_data.append({
            'Image_ID': image_id,
            'Tooth_Area': tooth_area,
            'Root_Area': root_area
        })

    # Create a pandas DataFrame from the collected data
    df_mask_areas = pd.DataFrame(mask_area_data)

    # Define the output file path
    output_csv_path = '/content/GT_mask_areas.csv'

    # Save the DataFrame to a CSV file
    df_mask_areas.to_csv(output_csv_path, index=False)

    print(f"\nMask areas calculated and saved to {output_csv_path}")
    display(df_mask_areas.head()) # Display the first few rows of the DataFrame

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# Directory containing the predicted masks
predicted_mask_directory = '/content/content/Dataset_split_train_test_266/test/predicted_masks'

# Get a list of predicted mask file paths
predicted_mask_paths = sorted(glob(os.path.join(predicted_mask_directory, '*.png')))

if not predicted_mask_paths:
    print(f"Error: No predicted mask files found in the directory: {predicted_mask_directory}")
else:
    print(f"Found {len(predicted_mask_paths)} predicted mask files.")

    # List to store results
    predicted_mask_area_data = []

    # Iterate through each predicted mask file
    for mask_path in predicted_mask_paths:
        # Get the filename (which can be used as image ID)
        mask_filename = os.path.basename(mask_path)
        image_id = os.path.splitext(mask_filename)[0] # Use filename without extension as ID

        # Read the mask
        # Use cv2.IMREAD_UNCHANGED to read as is, then ensure it's grayscale
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

        # If the mask is not grayscale, convert it
        if len(mask.shape) == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        # Assuming class ID 1 is 'Tooth' and class ID 2 is 'Root' in predicted masks
        tooth_area = np.sum(mask == 1)
        root_area = np.sum(mask == 2)

        # Append data to the list
        predicted_mask_area_data.append({
            'Image_ID': image_id,
            'Predicted_Tooth_Area': tooth_area,
            'Predicted_Root_Area': root_area
        })

    # Create a pandas DataFrame from the collected data
    df_predicted_mask_areas = pd.DataFrame(predicted_mask_area_data)

    # Define the output file path
    output_csv_path = '/content/Unet_mask_areas.csv'

    # Save the DataFrame to a CSV file
    df_predicted_mask_areas.to_csv(output_csv_path, index=False)

    print(f"\nPredicted mask areas calculated and saved to {output_csv_path}")
    display(df_predicted_mask_areas.head()) # Display the first few rows of the DataFrame

In [ ]:
# prompt: '/content/GT_mask_areas.csv'와 '/content/Unet_mask_areas.csv'파일을 '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare_0729_best'폴더 아래로 복사해

import os

# Define the destination directory in Google Drive
destination_dir = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare_0729_best'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Copy the files to the destination directory
!cp /content/GT_mask_areas.csv "{destination_dir}"
!cp /content/Unet_mask_areas.csv "{destination_dir}"

print(f"Copied GT_mask_areas.csv and Unet_mask_areas.csv to {destination_dir}")